# KYC OCR pipeline v3 — text-first architecture

Complete, runnable, fully offline. Edit **CELL 4** (`MODEL_PATH`, `ZIP_PATH`,
`CUSTOMER_DATABASE_PATH`, `OUTPUT_DIR`) and run top to bottom.

---

## Why the previous version returned mostly null — diagnosis before redesign

Four causes, ranked. Only the last is about resolution; the first two are about how the model was
asked, and they are the reason "fields a human can read" came back empty.

### 1. The prompt handed the model an all-null answer template

The previous prompt specified the output shape like this:

```json
{"surname_latin": {"value": null, "confidence": "unreadable"}, ...}
```

Copying that template verbatim is a valid, well-formed, zero-risk completion — and it is entirely
null. Every page got shown the failure answer before it started. **Fix:** the schema is now
described as a key list with a *filled* worked example; no null-filled template is ever shown.

### 2. "One illegible character ⇒ the whole value is null" made refusal the safe move

That rule is correct as a *policy* but catastrophic as an *instruction to a generator*: the model
cannot be partially right, so silence dominates. **Fix — the core architectural change:** the model
is no longer asked to decide nullability. It is asked to transcribe what it sees, marking
individual illegible characters with `?`. Python then applies the policy: a value containing `?`
is not emitted as a value, but the partial evidence is kept and reported. Anti-hallucination is
unchanged — the model still never invents — but we now capture `MOHA?ED` instead of nothing,
which is both more useful and more auditable.

### 3. Direct schema extraction is the wrong first task for a degraded scan

"Fill these 13 fields" asks for reading, layout understanding and judgement at once. "Transcribe
every line of text you can see" asks for the one thing a VLM does best. **Fix:** two passes over
the same pixels — PASS A transcribes lines verbatim, PASS B fills the schema — then Python maps
the lines to fields using multilingual labels. Agreement between the two independent passes
becomes an *objective* confidence signal (§28) rather than a self-report.

### 4. Resolution: the page was downscaled before the model ever saw it

At `MAX_IMAGE_DIMENSION = 1400`, a 300 dpi A4 page is scaled by 0.40. MRZ characters (3.2 mm)
arrive ~15 px tall; small print fares no better. Raising `PDF_RENDER_DPI` does **nothing** here —
final glyph size depends only on the max dimension. **Fix:** tiles and regions of interest are
re-rendered *from the PDF* at high DPI instead of upscaling discarded pixels.

A check worth recording: the 13-field schema including the MRZ costs ~246 output tokens, against
the old 320 cap. It fits, so truncation was **not** the main cause — but the 74-token margin
disappears with a long issuing authority or an Arabic value, so the MRZ is now removed from the
general schema call and given its own pass anyway.

---

## Architecture

```
PDF --pymupdf--> render EVERY page at PDF_RENDER_DPI (kept at full resolution)
   |
   +-- analyse page (CV, ~30 ms, no GPU): quality, orientation, blank, MRZ band, tiles
   |
   +-- PASS A  transcribe_lines()   full page  -> verbatim text lines, ? for illegible chars
   +-- PASS B  extract_fields()     full page  -> schema, worked example, no null template
   |              |
   |              +-- Python maps PASS A lines to fields via multilingual labels
   |              +-- agreement(A, B) = objective confidence
   |
   +-- PASS C  targeted, ONLY for fields still missing (MAX_TARGETED_RETRIES):
   |              MRZ band / name region / tile, re-rendered from the PDF at high DPI
   |
   +-- validate (format, check digits) -- flags only, never rewrites
   |
   +-- diagnose failures: A bad render / B low resolution / C bad crop / D preprocessing /
                          E image input / F generation / G model limit
```

Stage A (extraction from pixels) and Stage B (verification against other documents and the
database) are strictly separated. Nothing in Stage B can write into Stage A.

In [ ]:
# =========================================================================
# CELL 2 — ENVIRONMENT DIAGNOSTICS
# =========================================================================
import os, sys, platform, subprocess, importlib, json

# Must be set before torch is imported anywhere in this process.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

MODEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B-FP8/main"

print(f"{'python':<18}: {platform.python_version()}")
print(f"{'platform':<18}: {platform.platform()}")
for _m in ("torch", "transformers", "accelerate", "numpy", "pandas", "PIL", "cv2", "pymupdf"):
    try:
        _mod = importlib.import_module(_m)
        print(f"{_m:<18}: {getattr(_mod, '__version__', 'present')}")
    except Exception:
        print(f"{_m:<18}: NOT INSTALLED")

try:
    import torch
    print(f"{'cuda (torch)':<18}: {torch.version.cuda} | available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f"{'gpu[' + str(i) + ']':<18}: {p.name} | {p.total_memory/2**30:.1f} GiB | "
                  f"sm_{p.major}{p.minor}")
except Exception as exc:
    print("torch unavailable:", exc)

print(f"{'model path':<18}: {MODEL_PATH}")
print(f"{'model exists':<18}: {os.path.isdir(MODEL_PATH)}")
_cfg = os.path.join(MODEL_PATH, "config.json")
if os.path.isfile(_cfg):
    with open(_cfg) as fh:
        _raw = json.load(fh)
    print(f"{'model_type':<18}: {_raw.get('model_type')}")
    print(f"{'architectures':<18}: {_raw.get('architectures')}")
    print(f"{'vision tower':<18}: {'yes' if 'vision_config' in _raw else 'NO -- see CELL 7'}")
else:
    print("!! config.json not found at MODEL_PATH")

In [ ]:
# =========================================================================
# CELL 3 — IMPORTS
# =========================================================================
from __future__ import annotations

import io, re, gc, math, time, zipfile, hashlib, difflib, logging, unicodedata, traceback
from collections import OrderedDict, defaultdict
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone, date
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image

Image.MAX_IMAGE_PIXELS = None          # 300+ dpi A4 scans exceed Pillow's default guard

# PyMuPDF via its current import name only. The deprecated `fitz` alias is not used anywhere in
# this notebook, and no code path falls back to it.
try:
    import pymupdf
except Exception:
    pymupdf = None

try:
    import cv2
except Exception:
    cv2 = None

try:
    import pytesseract
    from pytesseract import Output as TessOutput
    pytesseract.get_tesseract_version()
except Exception:
    pytesseract, TessOutput = None, None

try:
    import torch
except Exception:
    torch = None

# NOTE: flash_attn is deliberately NOT imported, probed or installed anywhere in this notebook.
# The Domino environment cannot build it. CELL 6 selects a native PyTorch backend instead.

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-7s %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("kyc")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
utcnow = lambda: datetime.now(timezone.utc).isoformat()
print("run id:", RUN_ID)

In [ ]:
# =========================================================================
# CELL 4 — CONFIGURATION  (the only cell you normally edit)
# =========================================================================
@dataclass
class Config:
    # ---------------- paths -----------------------------------------------
    MODEL_PATH: str = MODEL_PATH
    ZIP_PATH: Path = Path("/domino/datasets/local/kyc/kyc_documents.zip")
    CUSTOMER_DATABASE_PATH: Optional[Path] = Path("/domino/datasets/local/kyc/customers.csv")
    OUTPUT_DIR: Path = Path("/domino/datasets/local/kyc/output_v3")

    # ---------------- rendering and resolution ----------------------------
    # Pages are rendered at this DPI and KEPT at full resolution. Downscaling happens per view,
    # never once destructively for the whole pipeline.
    PDF_RENDER_DPI: int = 300
    # Longest side of the FULL-PAGE view sent to the model. Qwen-VL spends roughly one visual
    # token per 28x28 px block, so 1600 px ~= 2300 tokens for a portrait page.
    MAX_IMAGE_DIMENSION: int = 1600
    MAX_VISUAL_TOKENS: int = 2400          # hard cap handed to the processor as max_pixels
    # Minimum glyph height for reliable OCR. Below this a region is escalated to a high-DPI
    # re-render rather than being sent as-is.
    MIN_TEXT_HEIGHT: float = 16.0
    MRZ_UPSCALE_FACTOR: float = 3.0
    HANDWRITING_UPSCALE_FACTOR: float = 2.5
    REGION_RENDER_DPI: int = 600           # ROI re-render: true resolution from the PDF
    MRZ_MAX_WIDTH: int = 1800
    MRZ_MIN_CHAR_HEIGHT: float = 22.0        # below this the MRZ ladder escalates
    MRZ_CANDIDATES_PER_PAGE: int = 3         # examined on EVERY page, then ranked globally
    MRZ_MIN_SCORE: float = 0.42              # below this a candidate is not an MRZ at all
    REGION_MAX_WIDTH: int = 1600
    PAGE_TILES: int = 2                    # tiles used when a page needs more effective resolution
    MAX_PAGES_PER_PDF: int = 12

    # ---------------- preprocessing (applied only when measured as needed) ----
    ENABLE_ORIENTATION_CORRECTION: bool = True
    ENABLE_DESKEW: bool = True
    ENABLE_CONTRAST_ENHANCEMENT: bool = True
    ENABLE_SHARPENING: bool = True         # mild unsharp, ROI crops only
    ENABLE_DENOISE: bool = True
    ENABLE_ADAPTIVE_THRESHOLD: bool = False  # last-resort MRZ variant only
    DESKEW_MIN_DEG: float = 0.4
    DESKEW_MAX_DEG: float = 15.0
    BLUR_VAR_POOR: float = 80.0            # variance of Laplacian at a fixed 1000 px work size
    BLUR_VAR_GOOD: float = 300.0
    CONTRAST_LOW: float = 0.30
    NOISE_SIGMA_HIGH: float = 6.0
    ILLUMINATION_POOR: float = 0.14
    INK_RATIO_BLANK: float = 0.002

    # ---------------- inference -------------------------------------------
    ATTENTION_IMPLEMENTATION: str = "sdpa"   # "sdpa" | "eager". FlashAttention is NEVER used.
    TORCH_DTYPE: str = "bfloat16"            # compute dtype around the FP8 weights
    # ---- FP8 (FineGrainedFP8Config) ----
    USE_FP8: bool = True
    FP8_ACTIVATION_SCHEME: str = "dynamic"   # the only scheme the class currently supports
    FP8_WEIGHT_BLOCK_SIZE: Tuple[int, int] = (128, 128)
    # Keep the VISION tower and the LM head out of FP8. Quantising the vision encoder costs
    # exactly the fine-detail fidelity that MRZ and handwriting OCR depend on, and the encoder is
    # a small fraction of the weights, so the memory saved is not worth the accuracy.
    FP8_SKIP_MODULES: Tuple[str, ...] = ("lm_head", "visual", "vision_tower", "merger",
                                         "vision_model", "multi_modal_projector")
    DEVICE_MAP: str = "auto"
    TEMPERATURE: float = 0.0                 # expressed as do_sample=False (see CELL 19)
    MAX_NEW_TOKENS: int = 512                # PASS B schema, MRZ excluded from that call
    MAX_NEW_TOKENS_LINES: int = 768          # PASS A line transcription needs room
    MAX_NEW_TOKENS_TARGETED: int = 160       # PASS C: few fields, so few tokens
    MAX_NEW_TOKENS_MRZ: int = 160
    MAX_NEW_TOKENS_TITLE: int = 128
    REPETITION_PENALTY: float = 1.0          # MUST be 1.0 -- see CELL 19
    NO_REPEAT_NGRAM_SIZE: int = 0            # MUST be 0 -- would forbid "<<" in an MRZ
    DISABLE_THINKING: bool = True
    RESERVE_VRAM_GIB: float = 8.0
    CPU_OFFLOAD_GIB: float = 64.0
    OOM_DOWNSCALE_FACTOR: float = 0.65
    OOM_MAX_DOWNSCALES: int = 2
    MOCK_MODEL: bool = False                 # True = rehearse the whole pipeline with no GPU

    # ---------------- retries ---------------------------------------------
    MAX_TARGETED_RETRIES: int = 2            # per field group, hard limit
    MAX_MRZ_ATTEMPTS: int = 3

    # ---------------- verification ----------------------------------------
    # Only these four database columns are used. Nothing else is read.
    DB_COL_ID: str = "Id tiers"
    DB_COL_NAME: str = "Nom abrégé tiers"
    DB_COL_DOB: str = "Date de naissance"
    DB_COL_EXPIRY: str = "Date d'expiration du Document"
    NAME_FUZZY_THRESHOLD: float = 0.90       # exposed on every comparison, never auto-promoted
    TITLE_MATCH_THRESHOLD: float = 0.72

    # ---------------- run control -----------------------------------------
    SAVE_DEBUG_IMAGES: bool = False          # True = keep images for successful pages too
    SAVE_DEBUG_ON_FAILURE: bool = True       # always keep the crop behind a failed field
    RESUME: bool = True
    LIMIT_CUSTOMERS: Optional[int] = None
    MAX_CONSECUTIVE_ERRORS: int = 3
    PROCESS_SECONDARY_DOCUMENTS: bool = True

    def dirs(self) -> Dict[str, Path]:
        d = {k: self.OUTPUT_DIR / v for k, v in {
            "pdfs": "00_selected_pdfs", "results": "01_results", "reports": "02_reports",
            "debug": "debug_images", "logs": "04_logs"}.items()}
        for p in d.values():
            p.mkdir(parents=True, exist_ok=True)
        return d


CFG = Config()
DIRS = CFG.dirs()

IDENTITY_DOC = "JUSTIFICATIF IDENTITE.PDF"
DOMICILE_DOC = "JUSTIFICATIF DOMICILE.PDF"
CONVENTION_DOC = "CONVENTION COMPTE.PDF"
FATCA_DOC = "FATCA.PDF"
SIGNATURE_DOC = "CARTON SIGNATURE.PDF"
DOCUMENTS_REQUIRED = [IDENTITY_DOC, DOMICILE_DOC, CONVENTION_DOC, FATCA_DOC, SIGNATURE_DOC]
PRESENCE_COLUMNS = OrderedDict([
    (IDENTITY_DOC, "identity_document_exists"),
    (DOMICILE_DOC, "domicile_document_exists"),
    (CONVENTION_DOC, "convention_compte_exists"),
    (FATCA_DOC, "fatca_exists"),
    (SIGNATURE_DOC, "signature_card_exists"),
])

IDENTITY_FIELDS = ["document_type", "surname_latin", "given_names_latin", "date_of_birth",
                   "place_of_birth", "nationality", "sex", "document_number", "issue_date",
                   "expiry_date", "issuing_authority", "personal_number", "mrz"]
# Mandatory when visually available (section 12)
MANDATORY_FIELDS = ["surname_latin", "given_names_latin", "date_of_birth", "expiry_date"]
NAME_FIELDS = ["surname_latin", "given_names_latin"]

print("model   :", CFG.MODEL_PATH)
print("zip     :", CFG.ZIP_PATH, "| exists:", CFG.ZIP_PATH.exists())
print("database:", CFG.CUSTOMER_DATABASE_PATH)
print("output  :", CFG.OUTPUT_DIR)

In [ ]:
# =========================================================================
# CELL 5 — DEPENDENCY CHECKS
# =========================================================================
def check_dependencies(cfg: Config = CFG) -> Dict[str, Any]:
    """Report what is present. The notebook degrades rather than crashing, and says how."""
    status: Dict[str, Any] = {}
    problems: List[str] = []

    status["pymupdf"] = pymupdf is not None
    if pymupdf is None:
        problems.append("pymupdf MISSING -- mandatory: PDF rendering and ROI re-rendering")
    status["opencv"] = cv2 is not None
    if cv2 is None:
        problems.append("opencv MISSING -- MRZ localisation, orientation and quality metrics "
                        "are disabled; OCR still runs with reduced recall")
    status["torch"] = torch is not None
    if torch is None and not cfg.MOCK_MODEL:
        problems.append("torch MISSING -- mandatory for inference (or set CFG.MOCK_MODEL=True)")
    status["tesseract"] = pytesseract is not None       # optional, orientation only
    status["cuda"] = bool(torch is not None and torch.cuda.is_available())

    try:
        import transformers
        status["transformers"] = transformers.__version__
    except Exception:
        status["transformers"] = None
        problems.append("transformers MISSING -- mandatory")

    status["model_path_exists"] = Path(cfg.MODEL_PATH).exists()
    if not status["model_path_exists"]:
        problems.append(f"MODEL_PATH does not exist: {cfg.MODEL_PATH}")
    status["zip_exists"] = Path(cfg.ZIP_PATH).exists()
    if not status["zip_exists"]:
        problems.append(f"ZIP_PATH does not exist: {cfg.ZIP_PATH}")
    status["database_exists"] = bool(cfg.CUSTOMER_DATABASE_PATH and
                                     Path(cfg.CUSTOMER_DATABASE_PATH).exists())
    if not status["database_exists"]:
        problems.append("customer database not found -- database verification will report "
                        "NOT_AVAILABLE for every customer (extraction is unaffected)")

    for k, v in status.items():
        print(f"  {k:<20}: {v}")
    if problems:
        print("\nATTENTION")
        for p in problems:
            print("  -", p)
    else:
        print("\nall checks passed")
    return status


DEP_STATUS = check_dependencies(CFG)

In [ ]:
# =========================================================================
# CELL 6 — GPU AND ATTENTION DIAGNOSTICS
# =========================================================================
def gpu_memory() -> Dict[str, float]:
    if torch is None or not torch.cuda.is_available():
        return {}
    out = {}
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        out[f"gpu{i}_total_gib"] = round(total / 2**30, 1)
        out[f"gpu{i}_free_gib"] = round(free / 2**30, 1)
        out[f"gpu{i}_allocated_gib"] = round(torch.cuda.memory_allocated(i) / 2**30, 1)
    return out


def gpu_report() -> None:
    """This process's memory AND other processes on the card. A second PID here explains most
    'OOM on every call' reports."""
    if torch is None or not torch.cuda.is_available():
        print("no CUDA device visible -- set CFG.MOCK_MODEL = True to rehearse")
        return
    print(f"device            : {torch.cuda.get_device_name(0)}")
    for k, v in gpu_memory().items():
        print(f"{k:<18}: {v} GiB")
    try:
        out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory",
                              "--format=csv,noheader"], capture_output=True, text=True,
                             timeout=15).stdout.strip()
        print("processes on GPU  :", out or "(none)")
        print("this pid          :", os.getpid())
    except Exception as exc:
        print("nvidia-smi unavailable:", exc)


def select_attention_implementation(cfg: Config = CFG) -> str:
    """Choose a NATIVE PyTorch attention backend.

    flash_attn is never imported, probed or installed: the Domino image cannot build it, and a
    missing optional import must never be a failure path. SDPA is PyTorch's own fused attention
    (memory-efficient / flash kernels where the hardware supports them) and needs no extra
    package. If the installed transformers build rejects sdpa for this architecture, fall back to
    eager, which always works."""
    requested = (cfg.ATTENTION_IMPLEMENTATION or "sdpa").lower()
    if requested == "flash_attention_2":
        log.warning("ATTENTION_IMPLEMENTATION=flash_attention_2 requested but flash_attn is not "
                    "available in this environment; using sdpa instead")
        requested = "sdpa"
    if requested == "sdpa":
        if torch is None:
            return "eager"
        if not hasattr(torch.nn.functional, "scaled_dot_product_attention"):
            log.warning("torch has no scaled_dot_product_attention (torch<2.0) -> eager")
            return "eager"
    return requested


def release_cuda_cache() -> None:
    """NOT called per page: emptying the cache forces re-allocation and costs performance.
    Reserved for OOM recovery and teardown."""
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


gpu_report()
ATTENTION_IMPL = select_attention_implementation(CFG)
print("attention backend :", ATTENTION_IMPL, "(flash_attn is never used)")

In [ ]:
# =========================================================================
# CELL 7 — TRANSFORMERS API CHECK AND FineGrainedFP8Config RESOLUTION
# =========================================================================
# IMPORTANT, verified against the installed package rather than assumed:
#
#   from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config   -> ImportError
#
# That module defines the RUNTIME pieces (FP8Linear, replace_with_fp8_linear, the Triton
# w8a8 block matmul). The CONFIG class lives in transformers.utils.quantization_config and is
# re-exported at the transformers top level. This resolver tries the path named in the spec
# first, then the ones that actually exist, and reports which succeeded -- so a future
# transformers release that does move the class keeps working.
import inspect

FP8_IMPORT_CANDIDATES = [
    ("transformers.integrations.finegrained_fp8", "FineGrainedFP8Config"),   # as specified
    ("transformers", "FineGrainedFP8Config"),                                # actual export
    ("transformers.utils.quantization_config", "FineGrainedFP8Config"),      # definition site
]


def resolve_fp8_config_class() -> Tuple[Any, str, List[str]]:
    """Return (class, import_path_used, constructor parameter names)."""
    import importlib
    errors = []
    for module_name, attr in FP8_IMPORT_CANDIDATES:
        try:
            mod = importlib.import_module(module_name)
            cls = getattr(mod, attr)
            params = [p for p in inspect.signature(cls.__init__).parameters
                      if p not in ("self", "args", "kwargs")]
            return cls, f"{module_name}.{attr}", params
        except Exception as exc:
            errors.append(f"{module_name}: {type(exc).__name__}: {exc}")
    raise ImportError("FineGrainedFP8Config not found. Tried:\n  " + "\n  ".join(errors) +
                      "\nFP8 quantisation is required by this pipeline; it will not silently "
                      "fall back to the non-quantised model.")


import transformers as _tf
print(f"{'transformers':<24}: {_tf.__version__}")
print(f"{'torch':<24}: {torch.__version__ if torch else 'MISSING'}")

FP8_CONFIG_CLASS, FP8_IMPORT_PATH, FP8_PARAMS = resolve_fp8_config_class()
print(f"{'FineGrainedFP8Config':<24}: FOUND via {FP8_IMPORT_PATH}")
print(f"{'constructor parameters':<24}: {FP8_PARAMS}")
print(f"{'docstring':<24}: {' '.join((FP8_CONFIG_CLASS.__doc__ or '').split())[:160]}...")

# The FP8 runtime pieces are in the integrations module; confirm they are importable.
try:
    from transformers.integrations.finegrained_fp8 import FP8Linear
    print(f"{'FP8Linear runtime':<24}: available")
except Exception as exc:
    FP8Linear = None
    print(f"{'FP8Linear runtime':<24}: NOT available ({exc})")

if torch is not None and torch.cuda.is_available():
    _cap = torch.cuda.get_device_capability(0)
    print(f"{'compute capability':<24}: sm_{_cap[0]}{_cap[1]} "
          f"({'FP8 tensor cores present' if _cap >= (8, 9) else 'NO native FP8 (needs sm_89+)'})")

In [ ]:
# =========================================================================
# CELL 8 — FP8 CONFIGURATION
# =========================================================================
def build_fp8_config(cfg: Config = CFG) -> Tuple[Any, Dict[str, Any]]:
    """Construct FineGrainedFP8Config using ONLY parameters the installed class accepts."""
    kwargs: Dict[str, Any] = {}
    if "activation_scheme" in FP8_PARAMS:
        kwargs["activation_scheme"] = cfg.FP8_ACTIVATION_SCHEME
    if "weight_block_size" in FP8_PARAMS and cfg.FP8_WEIGHT_BLOCK_SIZE:
        kwargs["weight_block_size"] = tuple(cfg.FP8_WEIGHT_BLOCK_SIZE)
    if "modules_to_not_convert" in FP8_PARAMS and cfg.FP8_SKIP_MODULES:
        # Vision tower and LM head stay out of FP8: the encoder is what resolves MRZ glyphs and
        # handwriting strokes, and it is a small share of the parameters.
        kwargs["modules_to_not_convert"] = list(cfg.FP8_SKIP_MODULES)
    quant_config = FP8_CONFIG_CLASS(**kwargs)
    print("FineGrainedFP8Config constructed with:", json.dumps(kwargs, default=str))
    return quant_config, kwargs


def checkpoint_quantization(cfg: Config = CFG) -> Optional[Dict[str, Any]]:
    """Read any quantization_config already present in the checkpoint's config.json."""
    p = Path(cfg.MODEL_PATH) / "config.json"
    if not p.exists():
        return None
    try:
        return json.loads(p.read_text()).get("quantization_config")
    except Exception:
        return None


FP8_QUANT_CONFIG, FP8_KWARGS = (build_fp8_config(CFG) if CFG.USE_FP8 else (None, {}))
CKPT_QUANT = checkpoint_quantization(CFG)
print("checkpoint quantization_config:", json.dumps(CKPT_QUANT) if CKPT_QUANT else "none declared")
if CKPT_QUANT and str(CKPT_QUANT.get("quant_method", "")).lower() not in ("fp8", "finegrained_fp8"):
    log.warning("checkpoint declares quant_method=%r, which is not FP8",
                CKPT_QUANT.get("quant_method"))

In [ ]:
# =========================================================================
# CELL 9 — MODEL AND PROCESSOR LOADING  (exactly once per kernel)
# =========================================================================
# Processor and chat-template mechanism are unchanged from the previous working notebook:
#   AutoProcessor (one instance, min_pixels/max_pixels)
#   processor.apply_chat_template(messages, add_generation_prompt=True) with content
#     [{"type": "image"}, {"type": "text", "text": ...}]
#   processor(text=[...], images=[PIL.Image], return_tensors="pt")
#   model.generate(...) inside torch.inference_mode()
# What changed: the weights are FP8 via FineGrainedFP8Config, and attention is sdpa.
VISION_HINTS = ("vl", "vision", "image_text", "qwen2_vl", "qwen2_5_vl", "qwen3_vl")


class QwenFP8Engine:
    _instance: Optional["QwenFP8Engine"] = None

    def __init__(self, cfg: Config = CFG):
        from transformers import AutoConfig, AutoProcessor
        import transformers as tf
        self.cfg = cfg
        assert torch is not None, "PyTorch is required"
        assert Path(cfg.MODEL_PATH).exists(), f"model path not found: {cfg.MODEL_PATH}"

        self.mem_before = gpu_memory()
        busy = max((v for k, v in self.mem_before.items() if k.endswith("allocated_gib")),
                   default=0.0)
        if busy > 1.0:
            log.warning("%.1f GiB already allocated on this GPU. If you re-ran this cell, call "
                        "free_model() or restart the kernel: two resident copies of the weights "
                        "will OOM on every generate().", busy)

        self.hf_config = AutoConfig.from_pretrained(cfg.MODEL_PATH, trust_remote_code=True,
                                                    local_files_only=True)
        archs = list(getattr(self.hf_config, "architectures", []) or [])
        mtype = str(getattr(self.hf_config, "model_type", "")).lower()
        if not (hasattr(self.hf_config, "vision_config") or
                any(h in " ".join(archs).lower() or h in mtype for h in VISION_HINTS)):
            raise RuntimeError(
                f"{cfg.MODEL_PATH}\n  model_type={mtype!r} architectures={archs}\n"
                "  No vision tower: this checkpoint cannot read an image.")

        # ---- processor: ONE instance, visual-token budget baked in --------
        px = cfg.MAX_VISUAL_TOKENS * 28 * 28      # Qwen-VL: ~1 visual token per 28x28 px block
        t0 = time.perf_counter()
        try:
            self.processor = AutoProcessor.from_pretrained(
                cfg.MODEL_PATH, trust_remote_code=True, local_files_only=True,
                min_pixels=256 * 28 * 28, max_pixels=px)
        except TypeError:
            self.processor = AutoProcessor.from_pretrained(cfg.MODEL_PATH, trust_remote_code=True,
                                                           local_files_only=True)
        self.processor_load_s = round(time.perf_counter() - t0, 2)
        self.tokenizer = getattr(self.processor, "tokenizer", self.processor)
        try:
            self.tokenizer.padding_side = "left"
        except Exception:
            pass

        kwargs: Dict[str, Any] = dict(
            torch_dtype=getattr(torch, cfg.TORCH_DTYPE), device_map=cfg.DEVICE_MAP,
            trust_remote_code=True, local_files_only=True, low_cpu_mem_usage=True,
            attn_implementation=ATTENTION_IMPL)
        if cfg.USE_FP8 and FP8_QUANT_CONFIG is not None:
            kwargs["quantization_config"] = FP8_QUANT_CONFIG

        last, self.model, self.fp8_path = None, None, None
        for cls_name in archs + ["AutoModelForImageTextToText", "AutoModelForVision2Seq"]:
            if not hasattr(tf, cls_name):
                continue
            for attempt in ("explicit_fp8_config", "checkpoint_fp8_config"):
                try:
                    t0 = time.perf_counter()
                    self.model = getattr(tf, cls_name).from_pretrained(cfg.MODEL_PATH, **kwargs)
                    self.loader = cls_name
                    self.load_s = round(time.perf_counter() - t0, 1)
                    self.fp8_path = attempt
                    break
                except Exception as exc:
                    msg = str(exc).lower()
                    last = f"{cls_name}/{attempt}: {type(exc).__name__}: {exc}"
                    release_cuda_cache()
                    # A pre-quantised checkpoint can refuse an explicit quantization_config. That
                    # is NOT a fallback to the non-quantised model: the checkpoint's own
                    # FineGrainedFP8Config applies, and it is verified below.
                    if attempt == "explicit_fp8_config" and "quantization_config" in kwargs and \
                            ("already" in msg or "quantiz" in msg):
                        log.warning("checkpoint rejected an explicit quantization_config (%s); "
                                    "loading with the checkpoint's own FP8 config", type(exc).__name__)
                        kwargs.pop("quantization_config", None)
                        continue
                    if "attention" in msg and kwargs.get("attn_implementation") != "eager":
                        log.warning("attn_implementation=%s rejected; retrying with eager",
                                    kwargs["attn_implementation"])
                        kwargs["attn_implementation"] = "eager"
                        continue
                    break
            if self.model is not None:
                break
        if self.model is None:
            raise RuntimeError(f"could not load the FP8 model. last error: {last}")

        self.attn = kwargs.get("attn_implementation", ATTENTION_IMPL)
        self.model.eval()
        gen_cfg = getattr(self.model, "generation_config", None)
        if gen_cfg is not None:        # neutralise any sampling defaults in the checkpoint
            gen_cfg.do_sample = False
            gen_cfg.temperature = gen_cfg.top_p = gen_cfg.top_k = None

        # ---- verify FP8 is ACTUALLY applied; never accept a silent bf16 load ----
        self.quantization = self._describe_quantization()
        if cfg.USE_FP8 and not self.quantization["is_fp8"]:
            raise RuntimeError(
                "FP8 was requested but the loaded model is not FP8 quantised "
                f"({self.quantization}). Refusing to continue on the non-quantised path.")
        self.mem_after = gpu_memory()
        self.name = f"{Path(cfg.MODEL_PATH).parent.name} [{self.loader}]"
        self.consecutive_ooms = 0

    def _describe_quantization(self) -> Dict[str, Any]:
        qc = getattr(self.model.config, "quantization_config", None)
        method = None
        if qc is not None:
            method = getattr(qc, "quant_method", None) or (
                qc.get("quant_method") if isinstance(qc, dict) else None)
            method = getattr(method, "value", method)
        n_fp8, n_linear, dtypes = 0, 0, {}
        for _, module in self.model.named_modules():
            cname = type(module).__name__
            if cname == "FP8Linear":
                n_fp8 += 1
            elif cname == "Linear":
                n_linear += 1
        for _, p in list(self.model.named_parameters())[:400]:
            dtypes[str(p.dtype)] = dtypes.get(str(p.dtype), 0) + 1
        return {"quant_method": str(method) if method else None,
                "n_fp8_linear_modules": n_fp8, "n_plain_linear_modules": n_linear,
                "parameter_dtypes_sample": dtypes,
                "is_fp8": bool(n_fp8 > 0 or (method and "fp8" in str(method).lower()))}

    @classmethod
    def get(cls, cfg: Config = CFG) -> "QwenFP8Engine":
        if cls._instance is None:
            cls._instance = cls(cfg)
        return cls._instance


class MockEngine:
    """Rehearsal without a GPU. Transcribes nothing: it invents nothing."""
    name, attn, consecutive_ooms = "MOCK", "n/a", 0
    quantization = {"is_fp8": False, "quant_method": "mock"}
    load_s = processor_load_s = 0.0
    mem_before = mem_after = {}


def free_model() -> None:
    inst = QwenFP8Engine._instance
    if inst is not None:
        try:
            inst.model.to("meta")
        except Exception:
            pass
        del inst.model
        QwenFP8Engine._instance = None
    release_cuda_cache()
    log.info("model freed. GPU: %s", gpu_memory() or "no CUDA")


def load_model(cfg: Config = CFG):
    """Load model AND processor exactly once. Never call this inside a loop."""
    return MockEngine() if cfg.MOCK_MODEL else QwenFP8Engine.get(cfg)


ENGINE = load_model(CFG)
print("engine        :", ENGINE.name)
print("attention     :", ENGINE.attn)
print("quantization  :", json.dumps(ENGINE.quantization, default=str))
print("fp8 config via:", getattr(ENGINE, "fp8_path", "n/a"))
print("load times    : model %.1fs | processor %.1fs" %
      (getattr(ENGINE, "load_s", 0.0), getattr(ENGINE, "processor_load_s", 0.0)))

In [ ]:
# =========================================================================
# CELL 10 — MODEL MEMORY DIAGNOSTICS AND WARM-UP
# =========================================================================
# Warm-up matters for honest benchmarking: the first call pays for CUDA context creation, kernel
# autotuning and lazy weight materialisation. It is measured separately and EXCLUDED from the
# steady-state figures reported in the benchmark cell.
MEMORY_TRACE: List[Dict[str, Any]] = []


def record_memory(stage: str) -> Dict[str, Any]:
    rec = {"stage": stage, "timestamp": utcnow()}
    if torch is not None and torch.cuda.is_available():
        rec.update({
            "allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 2),
            "reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 2),
            "max_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2),
            "free_gib": round(torch.cuda.mem_get_info()[0] / 2**30, 2)})
    MEMORY_TRACE.append(rec)
    return rec


print("before load :", getattr(ENGINE, "mem_before", {}))
print("after load  :", record_memory("after_model_load"))

WARMUP_STATS: Dict[str, Any] = {"ran": False}


def warm_up(engine=None, cfg: Config = CFG) -> Dict[str, Any]:
    """One throwaway inference on a synthetic image, timed separately from real work."""
    engine = engine or ENGINE
    if isinstance(engine, MockEngine):
        return {"ran": False, "reason": "mock engine"}
    img = Image.new("RGB", (768, 512), "white")
    t0 = time.perf_counter()
    try:
        _ = run_qwen_ocr(img, "Return exactly: {\"ok\": true}", 16, engine, cfg=cfg)
        out = {"ran": True, "warmup_s": round(time.perf_counter() - t0, 2)}
    except Exception as exc:
        out = {"ran": False, "error": f"{type(exc).__name__}: {exc}"}
    out["memory"] = record_memory("after_warmup")
    return out


# warm_up() is called from the driver, AFTER run_qwen_ocr is defined.
print("CELL 10 ready: record_memory(), warm_up()")

In [ ]:
# =========================================================================
# CELL 8 — CUSTOMER DATABASE  (verification reference only; exactly four columns)
# =========================================================================
# The database is NEVER an OCR source and can never complete an unreadable value. Only these
# four columns are read; every other column in the file is ignored by construction.
def strip_accents(text: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", str(text))
                   if not unicodedata.combining(c))


def _find_column(columns: Sequence[str], wanted: str) -> Optional[str]:
    """Locate a column tolerating accent/case/whitespace differences in the export."""
    def key(s):
        return re.sub(r"[^a-z0-9]", "", strip_accents(str(s)).lower())
    target = key(wanted)
    for c in columns:
        if key(c) == target:
            return c
    for c in columns:                       # e.g. "Date d'expiration du Document (JJ/MM/AAAA)"
        if target and target in key(c):
            return c
    return None


def load_database(path: Optional[Path] = None, cfg: Config = CFG) -> Optional[pd.DataFrame]:
    path = Path(path or cfg.CUSTOMER_DATABASE_PATH) if (path or cfg.CUSTOMER_DATABASE_PATH) \
        else None
    if path is None or not path.exists():
        log.warning("customer database not found (%s): database verification -> NOT_AVAILABLE",
                    path)
        return None
    raw = pd.read_csv(path, dtype=str, sep=None, engine="python").fillna("") \
        if path.suffix.lower() == ".csv" else pd.read_excel(path, dtype=str).fillna("")

    mapping = {}
    for logical, wanted in [("customer_id", cfg.DB_COL_ID), ("name", cfg.DB_COL_NAME),
                            ("date_of_birth", cfg.DB_COL_DOB), ("expiry_date", cfg.DB_COL_EXPIRY)]:
        col = _find_column(raw.columns, wanted)
        if col is None:
            log.error("database column %r not found; available: %s", wanted, list(raw.columns))
            return None
        mapping[logical] = col
    log.info("database columns in use (and ONLY these): %s", mapping)

    df = raw[[mapping[k] for k in ("customer_id", "name", "date_of_birth", "expiry_date")]].copy()
    df.columns = ["customer_id", "name", "date_of_birth", "expiry_date"]
    df["customer_id"] = df["customer_id"].astype(str).str.strip()
    df = df.set_index("customer_id", drop=False)
    log.info("customer database: %d records", len(df))
    return df


def database_record(db: Optional[pd.DataFrame], customer_id: str) -> Optional[Dict[str, str]]:
    if db is None:
        return None
    cid = str(customer_id).strip()
    if cid not in db.index:
        return None
    row = db.loc[cid]
    if isinstance(row, pd.DataFrame):       # duplicate ids: refuse to pick one silently
        log.warning("customer %s appears %d times in the database; skipping comparison",
                    cid, len(row))
        return None
    return {k: str(v).strip() for k, v in row.to_dict().items()}


DB = load_database(cfg=CFG)
print("CELL 8 ready:", "database loaded" if DB is not None else "no database")

In [ ]:
# =========================================================================
# CELL 9 — ZIP EXTRACTION
# =========================================================================
def norm_name(text: str) -> str:
    """Accent-free, upper-case, punctuation-collapsed. Filename matching only."""
    s = re.sub(r"[^A-Z0-9]+", " ", strip_accents(text).upper())
    return re.sub(r"\s+", " ", s).strip()


DOC_ALIASES: "OrderedDict[str, List[str]]" = OrderedDict([
    (IDENTITY_DOC, ["JUSTIFICATIF IDENTITE", "JUSTIFICATIF D IDENTITE", "JUSTIF IDENTITE",
                    "PIECE IDENTITE", "PIECE D IDENTITE", "IDENTITE"]),
    (DOMICILE_DOC, ["JUSTIFICATIF DOMICILE", "JUSTIFICATIF DE DOMICILE", "JUSTIF DOMICILE",
                    "PREUVE DE DOMICILE", "DOMICILE"]),
    (CONVENTION_DOC, ["CONVENTION COMPTE", "CONVENTION DE COMPTE", "CONVENTION DU COMPTE",
                      "CONVENTION OUVERTURE COMPTE"]),
    (FATCA_DOC, ["FATCA", "FORMULAIRE FATCA", "FATCA CRS", "AUTOCERTIFICATION FATCA"]),
    # Earlier specifications spelled it SIGNATUTE; both are accepted so a folder matches
    # whichever way the file was actually named.
    (SIGNATURE_DOC, ["CARTON SIGNATURE", "CARTON SIGNATUTE", "CARTON DE SIGNATURE",
                     "SPECIMEN SIGNATURE", "SPECIMEN DE SIGNATURE", "SPICIMEN DE SIGNATURE"]),
])
DOC_NORM = {d: sorted({norm_name(a) for a in [d] + al}, key=len, reverse=True)
            for d, al in DOC_ALIASES.items()}
DOC_SLUG = {d: norm_name(d).replace(" ", "_") for d in DOC_ALIASES}


def match_document(filename: str, fuzzy_threshold: float = 0.88
                   ) -> Tuple[Optional[str], str, float]:
    """(canonical_name, match_type, score). Only capitalisation/whitespace/accents are treated as
    insignificant; an unrelated PDF must not be mapped, so fuzzy hits are flagged for review."""
    p = Path(str(filename))
    if p.suffix.lower() != ".pdf":
        return None, "not_pdf", 0.0
    stem = re.sub(r"\s+\d+$", "", norm_name(p.stem)).strip()
    for doc, aliases in DOC_NORM.items():
        if stem in aliases:
            return doc, "exact", 1.0
    for doc, aliases in DOC_NORM.items():
        for a in aliases:
            if len(a) >= 5 and a in stem:
                return doc, "contains", 0.95
    best_doc, best = None, 0.0
    for doc, aliases in DOC_NORM.items():
        for a in aliases:
            r = difflib.SequenceMatcher(None, stem, a).ratio()
            if r > best:
                best_doc, best = doc, r
    return (best_doc, "fuzzy", round(best, 3)) if best >= fuzzy_threshold \
        else (None, "no_match", 0.0)


def _safe_parts(member: str) -> Optional[List[str]]:
    name = member.replace("\\", "/")
    if name.startswith("/") or re.match(r"^[A-Za-z]:", name):
        return None
    parts = [p for p in name.split("/") if p not in ("", ".")]
    return None if any(p == ".." for p in parts) else parts


def extract_zip(zip_path: Path = None, cfg: Config = CFG) -> pd.DataFrame:
    """Unzip, keeping only the five required PDFs. The folder name is the customer_id and is
    carried through every downstream record."""
    zip_path = Path(zip_path or cfg.ZIP_PATH)
    assert zip_path.exists(), f"ZIP not found: {zip_path}"
    rows, seen = [], {}
    with zipfile.ZipFile(zip_path) as zf:
        infos = [i for i in zf.infolist() if not i.is_dir()]
        parts_all = [_safe_parts(i.filename) for i in infos]
        roots = {p[0] for p in parts_all if p and len(p) > 1}
        root = roots.pop() if len(roots) == 1 and any(p and len(p) > 2 for p in parts_all) else None
        for info, parts in zip(infos, parts_all):
            if parts is None:
                log.warning("unsafe zip member skipped: %s", info.filename)
                continue
            if root and parts[0] == root:
                parts = parts[1:]
            if not parts:
                continue
            customer_id = parts[0] if len(parts) > 1 else "_ARCHIVE_ROOT_"
            fname = parts[-1]
            doc, mtype, score = match_document(fname)
            row = dict(customer_id=customer_id, member="/".join(parts), filename=fname,
                       matched_document=doc or "", match_type=mtype, match_score=score,
                       size_bytes=info.file_size, extracted_path="")
            if doc is not None:
                dest_dir = DIRS["pdfs"] / customer_id
                dest_dir.mkdir(parents=True, exist_ok=True)
                n = seen.get((customer_id, doc), 0)
                seen[(customer_id, doc)] = n + 1
                dest = dest_dir / (f"{DOC_SLUG[doc]}.pdf" if n == 0
                                   else f"{DOC_SLUG[doc]}__dup{n+1}.pdf")
                dest.write_bytes(zf.read(info))
                row["extracted_path"] = str(dest)
            rows.append(row)
    catalog = pd.DataFrame(rows)
    log.info("zip: %d members, %d customer folders, %d required PDFs kept",
             len(catalog), catalog["customer_id"].nunique(),
             int((catalog["matched_document"] != "").sum()))
    return catalog


print("CELL 9 ready: extract_zip()")

In [ ]:
# =========================================================================
# CELL 10 — CUSTOMER / DOCUMENT DISCOVERY
# =========================================================================
@dataclass
class CustomerDocs:
    customer_id: str
    documents: Dict[str, Optional[str]]
    n_files: int

    @property
    def has_identity(self) -> bool:
        return bool(self.documents.get(IDENTITY_DOC))

    def path(self, doc: str) -> Optional[Path]:
        p = self.documents.get(doc)
        return Path(p) if p else None


def discover_documents(catalog: pd.DataFrame) -> List[CustomerDocs]:
    out = []
    for cid, grp in catalog.groupby("customer_id", sort=True):
        docs = {}
        for doc in DOCUMENTS_REQUIRED:
            hit = grp[(grp["matched_document"] == doc) & (grp["extracted_path"] != "")]
            docs[doc] = hit.iloc[0]["extracted_path"] if len(hit) else None
        out.append(CustomerDocs(str(cid), docs, len(grp)))
    return out


def select_targets(customers: List[CustomerDocs], cfg: Config = CFG) -> List[CustomerDocs]:
    """Only folders holding the identity document reach the model."""
    targets = [c for c in customers if c.has_identity]
    if cfg.LIMIT_CUSTOMERS:
        targets = targets[:cfg.LIMIT_CUSTOMERS]
    log.info("discovery: %d customers, %d with %s -> %d to process",
             len(customers), sum(c.has_identity for c in customers), IDENTITY_DOC, len(targets))
    return targets


print("CELL 10 ready: discover_documents()")

In [ ]:
# =========================================================================
# CELL 11 — DOCUMENT PRESENCE REPORT
# =========================================================================
def build_presence_report(customers: List[CustomerDocs], catalog: pd.DataFrame,
                          cfg: Config = CFG) -> pd.DataFrame:
    rows = []
    for c in customers:
        row: Dict[str, Any] = {"customer_id": c.customer_id}
        for doc, col in PRESENCE_COLUMNS.items():
            row[col] = "TRUE" if c.documents.get(doc) else "FALSE"
        grp = catalog[catalog["customer_id"] == c.customer_id]
        fuzzy = grp[grp["match_type"] == "fuzzy"]["filename"].tolist()
        row.update({"n_files_in_folder": c.n_files,
                    "n_required_present": sum(bool(c.documents.get(d))
                                              for d in DOCUMENTS_REQUIRED),
                    "will_be_processed": c.has_identity,
                    "fuzzy_matched_filenames": ";".join(fuzzy),
                    "needs_filename_review": bool(fuzzy)})
        rows.append(row)
    df = pd.DataFrame(rows)
    out = DIRS["reports"] / "document_presence_report.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig")     # BOM: Excel keeps accents and Arabic
    log.info("presence report -> %s", out)
    summary = pd.DataFrame({
        "present": [int((df[c] == "TRUE").sum()) for c in PRESENCE_COLUMNS.values()],
        "missing": [int((df[c] == "FALSE").sum()) for c in PRESENCE_COLUMNS.values()]},
        index=list(PRESENCE_COLUMNS))
    print(summary.to_string())
    return df


print("CELL 11 ready: build_presence_report()")

In [ ]:
# =========================================================================
# CELL 12 — PDF RENDERING WITH PYMUPDF
# =========================================================================
# Every page is rendered explicitly and KEPT at full resolution. The model never receives a PDF;
# it receives images this code produced, so "page 2 was processed" is verifiable, not assumed.
@dataclass
class RenderedPage:
    page_number: int
    image: Image.Image                 # full-resolution render, never downscaled in place
    width: int
    height: int
    render_dpi: int
    render_time: float
    pdf_rect: Tuple[float, float, float, float]
    native_image_px: Optional[Tuple[int, int]] = None


def open_pdf(pdf_path: Path):
    if pymupdf is None:
        raise RuntimeError("pymupdf is required (note: the deprecated `fitz` alias is not used)")
    doc = pymupdf.open(str(pdf_path))
    if doc.page_count == 0:
        doc.close()
        raise ValueError(f"PDF has zero pages: {pdf_path}")
    return doc


def _largest_embedded_image(doc, index: int) -> Optional[Tuple[int, int]]:
    """Native size of the biggest raster on the page.

    A scanned PDF is usually one full-page JPEG. If that JPEG is 900 px wide, rendering at 600 dpi
    only interpolates -- this tells the diagnostics whether more DPI could ever help."""
    try:
        infos = doc[index].get_images(full=True)
        return max(((i[2], i[3]) for i in infos), key=lambda wh: wh[0] * wh[1]) if infos else None
    except Exception:
        return None


def render_pdf_pages(pdf_path: Path, cfg: Config = CFG) -> List[RenderedPage]:
    """Render EVERY page independently. No resizing happens here."""
    doc = open_pdf(pdf_path)
    pages: List[RenderedPage] = []
    try:
        n = min(doc.page_count, cfg.MAX_PAGES_PER_PDF)
        if doc.page_count > cfg.MAX_PAGES_PER_PDF:
            log.warning("%s has %d pages; capped at %d", pdf_path.name, doc.page_count, n)
        for i in range(n):
            t0 = time.perf_counter()
            page = doc[i]
            pix = page.get_pixmap(dpi=cfg.PDF_RENDER_DPI, colorspace=pymupdf.csRGB, alpha=False)
            img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples).copy()
            r = page.rect
            pages.append(RenderedPage(i + 1, img, pix.width, pix.height, cfg.PDF_RENDER_DPI,
                                      round(time.perf_counter() - t0, 3),
                                      (r.x0, r.y0, r.x1, r.y1),
                                      _largest_embedded_image(doc, i)))
    finally:
        doc.close()
    log.info("%s: %d pages rendered (%s)", pdf_path.name, len(pages),
             ", ".join(f"p{p.page_number} {p.width}x{p.height} {p.render_time}s" for p in pages))
    return pages


def render_pdf_clip(pdf_path: Path, page_number: int,
                    clip_rect: Tuple[float, float, float, float],
                    dpi: int) -> Optional[Image.Image]:
    """Re-render ONE RECTANGLE of a page at high DPI, straight from the PDF.

    This is the core resolution fix. Upscaling a downscaled crop interpolates pixels that were
    already discarded; re-rendering the clip recovers real detail from the source, and a narrow
    strip costs a fraction of the visual tokens a full high-DPI page would."""
    if pymupdf is None:
        return None
    doc = None
    try:
        doc = pymupdf.open(str(pdf_path))
        page = doc[page_number - 1]
        rect = pymupdf.Rect(*clip_rect) & page.rect
        if rect.is_empty or rect.width < 1 or rect.height < 1:
            return None
        zoom = dpi / 72.0
        pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), clip=rect,
                              colorspace=pymupdf.csRGB, alpha=False)
        return Image.frombytes("RGB", (pix.width, pix.height), pix.samples).copy()
    except Exception as exc:
        log.warning("clip re-render failed p%d: %s", page_number, exc)
        return None
    finally:
        if doc is not None:
            doc.close()


print("CELL 12 ready: render_pdf_pages(), render_pdf_clip()  [pymupdf]")

In [ ]:
# =========================================================================
# CELL 13 — ORIENTATION CORRECTION
# =========================================================================
# A VLM call just to ask "which way up?" costs a full image prefill for four output tokens.
# This cascade costs ~20 ms: tesseract OSD if installed -> projection-profile axis test ->
# baseline-asymmetry test for the 180 degree case.
WORK_SIDE = 1000       # fixed measurement scale: blur variance is meaningless across sizes


def _work_gray(img: Image.Image, side: int = WORK_SIDE) -> np.ndarray:
    g = img.convert("L")
    if max(g.size) > side:
        s = side / max(g.size)
        g = g.resize((max(1, int(g.width * s)), max(1, int(g.height * s))), Image.BILINEAR)
    return np.asarray(g, dtype=np.uint8)


def _axis_score(binv: np.ndarray) -> float:
    """>1 = text lines run horizontally (0/180); <1 = vertically (90/270)."""
    row, col = binv.sum(axis=1).astype(np.float32), binv.sum(axis=0).astype(np.float32)
    rv = row.var() / (row.mean() ** 2 + 1e-6)
    cvv = col.var() / (col.mean() ** 2 + 1e-6)
    return float((rv + 1e-6) / (cvv + 1e-6))


def _updown_score(binv: np.ndarray) -> float:
    """Ink-mass asymmetry within text bands. Positive = upright.

    Capitals and ascenders outnumber descenders in Latin and Cyrillic, and Arabic also carries
    most of its mass above the baseline, so a band's ink centroid sits above its geometric
    centre when the page is upright."""
    row = binv.sum(axis=1).astype(np.float32)
    if row.max() <= 0:
        return 0.0
    thr = row.mean() + 0.3 * row.std()
    bands, start = [], None
    for i, v in enumerate(row):
        if v > thr and start is None:
            start = i
        elif v <= thr and start is not None:
            if i - start >= 3:
                bands.append((start, i))
            start = None
    if start is not None and len(row) - start >= 3:
        bands.append((start, len(row)))
    if len(bands) < 3:
        return 0.0
    scores = []
    for a, b in bands[:60]:
        seg = row[a:b]
        if seg.sum() <= 0:
            continue
        idx = np.arange(len(seg), dtype=np.float32)
        scores.append(0.5 - float((seg * idx).sum() / seg.sum()) / max(1, len(seg) - 1))
    return float(np.mean(scores) * 2.0) if scores else 0.0


def detect_orientation(img: Image.Image, cfg: Config = CFG) -> Dict[str, Any]:
    out = {"rotation": 0, "method": "disabled", "margin": 1.0, "osd_conf": None}
    if not cfg.ENABLE_ORIENTATION_CORRECTION or cv2 is None:
        return out
    if pytesseract is not None:
        try:
            small = img.copy()
            small.thumbnail((1000, 1000))
            osd = pytesseract.image_to_osd(small, output_type=TessOutput.DICT, config="--psm 0")
            conf = float(osd.get("orientation_conf", 0.0))
            out["osd_conf"] = conf
            if conf >= 2.0:
                return {**out, "rotation": int(osd.get("rotate", 0)) % 360,
                        "method": "tesseract_osd"}
        except Exception:
            pass
    gray = _work_gray(img, 800)
    binv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    axis = _axis_score(binv)
    candidates = (0, 180) if axis >= 1.0 else (90, 270)
    scored = {d: _updown_score(np.ascontiguousarray(np.rot90(binv, k=d // 90) if d else binv))
              for d in candidates}
    best = max(scored, key=lambda d: scored[d])
    out.update({"rotation": int(best), "method": "cv_projection",
                "margin": round(abs(scored[candidates[0]] - scored[candidates[1]]), 4),
                "axis_score": round(float(axis), 3)})
    return out


def correct_orientation(img: Image.Image, cfg: Config = CFG) -> Tuple[Image.Image, Dict[str, Any]]:
    info = detect_orientation(img, cfg)
    if info["rotation"]:
        img = img.rotate(info["rotation"], expand=True)      # PIL rotates counter-clockwise
    return img, info


print("CELL 13 ready: detect_orientation(), correct_orientation()")

In [ ]:
# =========================================================================
# CELL 14 — IMAGE PREPROCESSING  (modular; applied only when measured as needed)
# =========================================================================
def estimate_noise_sigma(gray: np.ndarray) -> float:
    """Immerkaer estimator: one 3x3 convolution; separates sensor noise from real structure."""
    if cv2 is None or gray.size == 0 or min(gray.shape) < 5:
        return 0.0
    M = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)
    conv = cv2.filter2D(gray.astype(np.float32), -1, M)
    h, w = gray.shape
    return round(float(np.abs(conv).sum() * math.sqrt(0.5 * math.pi) /
                       (6.0 * (w - 2) * (h - 2))), 2)


def estimate_text_height_px(gray: np.ndarray) -> float:
    """Median height of text-like connected components -- the metric that actually predicts OCR
    failure. Nominal DPI says nothing about how large the glyphs are."""
    if cv2 is None or gray.size == 0:
        return 0.0
    binv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    n, _, stats, _ = cv2.connectedComponentsWithStats(binv, connectivity=8)
    H = gray.shape[0]
    hs = [stats[i][3] for i in range(1, n)
          if 3 <= stats[i][3] <= max(8, H * 0.08) and 1 <= stats[i][2] <= H * 0.15
          and stats[i][4] >= 6]
    return round(float(np.median(hs)), 1) if len(hs) >= 8 else 0.0


def analyze_page_quality(rendered: RenderedPage, cfg: Config = CFG) -> Dict[str, Any]:
    """Measure the page. No GPU, ~30 ms. Drives preprocessing, tiling and diagnostics."""
    img = rendered.image
    gray = _work_gray(img)
    scale_back = max(img.size) / max(gray.shape) if max(gray.shape) else 1.0
    lap = float(cv2.Laplacian(gray, cv2.CV_64F).var()) if cv2 is not None else \
        float(np.diff(gray.astype(np.float32), axis=1).var())
    p5, p95 = np.percentile(gray, [5, 95])
    contrast = float((p95 - p5) / 255.0)
    noise = estimate_noise_sigma(gray)
    text_h_full = round(estimate_text_height_px(gray) * scale_back, 1)
    ink = 0.0
    if cv2 is not None:
        binv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
        ink = round(float((binv > 0).mean()), 4)
    g32 = gray.astype(np.float32)
    bh, bw = max(1, gray.shape[0] // 16), max(1, gray.shape[1] // 16)
    blocks = [g32[i:i + bh, j:j + bw].mean() for i in range(0, gray.shape[0] - bh + 1, bh)
              for j in range(0, gray.shape[1] - bw + 1, bw)]
    uniformity = round(float(np.std(blocks) / 255.0), 4) if blocks else 0.0

    blur_score = float(np.clip((lap - cfg.BLUR_VAR_POOR) /
                               max(1e-6, cfg.BLUR_VAR_GOOD - cfg.BLUR_VAR_POOR), 0, 1))
    contrast_score = float(np.clip(contrast / cfg.CONTRAST_LOW, 0, 1))
    noise_score = float(np.clip(1 - noise / max(1e-6, 2 * cfg.NOISE_SIGMA_HIGH), 0, 1))

    # Text height AFTER the page is scaled to MAX_IMAGE_DIMENSION: this is what the model
    # actually sees, and it is the number that decides whether tiling is required.
    view_scale = min(1.0, cfg.MAX_IMAGE_DIMENSION / max(rendered.width, rendered.height))
    text_h_view = round(text_h_full * view_scale, 1)

    q = {"page_number": rendered.page_number, "width": rendered.width, "height": rendered.height,
         "render_dpi": rendered.render_dpi,
         "native_image_px": list(rendered.native_image_px) if rendered.native_image_px else None,
         "text_height_px_full": text_h_full, "text_height_px_in_view": text_h_view,
         "blur_laplacian_var": round(lap, 1), "blur_score": round(blur_score, 3),
         "contrast_raw": round(contrast, 3), "contrast_score": round(contrast_score, 3),
         "brightness": round(float(gray.mean() / 255.0), 3),
         "noise_sigma": noise, "noise_score": round(noise_score, 3),
         "illumination_uniformity": uniformity, "ink_ratio": ink,
         "visual_clarity": round(0.4 * blur_score + 0.3 * contrast_score + 0.3 * noise_score, 3)}
    # Both conditions required: a small ID card on white A4 has a genuinely tiny ink ratio.
    q["is_blank"] = bool(ink < cfg.INK_RATIO_BLANK and text_h_full <= 0)
    q["needs_contrast_enhancement"] = bool(cfg.ENABLE_CONTRAST_ENHANCEMENT and
                                           (contrast < cfg.CONTRAST_LOW or
                                            uniformity > cfg.ILLUMINATION_POOR))
    q["needs_denoise"] = bool(cfg.ENABLE_DENOISE and noise > cfg.NOISE_SIGMA_HIGH)
    q["needs_tiling"] = bool(0 < text_h_view < cfg.MIN_TEXT_HEIGHT)
    q["quality"] = ("blank" if q["is_blank"] else
                    "good" if q["visual_clarity"] >= 0.75 else
                    "fair" if q["visual_clarity"] >= 0.45 else "poor")
    return q


def _clahe(img: Image.Image, clip: float = 2.0, tiles: Tuple[int, int] = (8, 8)) -> Image.Image:
    g = np.asarray(img.convert("L"))
    return Image.fromarray(cv2.createCLAHE(clipLimit=clip,
                                           tileGridSize=tiles).apply(g)).convert("RGB")


def _flatten_illumination(img: Image.Image) -> Image.Image:
    """Divide out a heavily blurred copy: removes shadows and vignetting without touching strokes."""
    g = np.asarray(img.convert("L"), dtype=np.float32)
    bg = cv2.GaussianBlur(g, (0, 0), sigmaX=max(g.shape) / 30.0)
    return Image.fromarray(np.clip(g / (bg + 1e-3) * float(np.median(bg)), 0, 255)
                           .astype(np.uint8)).convert("RGB")


def _unsharp(img: Image.Image, amount: float = 1.5) -> Image.Image:
    """Mild unsharp mask: raises edge contrast on an already high-resolution crop. It is a linear
    filter -- it does not manufacture glyph shapes, and it is never used to make an unreadable
    character 'readable'. Every applied operation is recorded with the result."""
    g = np.asarray(img.convert("L"))
    blur = cv2.GaussianBlur(g, (0, 0), 1.2)
    return Image.fromarray(cv2.addWeighted(g, amount, blur, 1 - amount, 0)).convert("RGB")


def _denoise(img: Image.Image) -> Image.Image:
    """Edge preserving ONLY. Median/Gaussian blur eats thin strokes and Arabic diacritics."""
    return Image.fromarray(cv2.bilateralFilter(np.asarray(img.convert("L")), 5, 45, 45)
                           ).convert("RGB")


def _resize_max(img: Image.Image, max_side: int) -> Image.Image:
    if max(img.size) <= max_side:
        return img
    s = max_side / max(img.size)
    return img.resize((max(1, int(img.width * s)), max(1, int(img.height * s))), Image.LANCZOS)


def preprocess_page(img: Image.Image, quality: Dict[str, Any], cfg: Config = CFG,
                    max_dimension: Optional[int] = None,
                    already_oriented: bool = False) -> Tuple[Image.Image, Dict[str, Any]]:
    """Geometry -> resize -> photometry ON THE SMALL IMAGE.

    Order matters for speed: filtering at full 8.7 MP and then downscaling spends seconds on
    pixels that are about to be discarded."""
    max_dimension = max_dimension or cfg.MAX_IMAGE_DIMENSION
    t0 = time.perf_counter()
    applied: List[str] = []
    meta: Dict[str, Any] = {"input_size": list(img.size), "rotation": 0, "skew_deg": 0.0}

    if not already_oriented:
        img, oinfo = correct_orientation(img, cfg)
        meta["orientation"] = oinfo
        meta["rotation"] = oinfo["rotation"]
        if oinfo["rotation"]:
            applied.append(f"rotate_{oinfo['rotation']}")

    if cfg.ENABLE_DESKEW and cv2 is not None:
        gray = _work_gray(img, 1000)
        binv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
        binv = cv2.dilate(binv, cv2.getStructuringElement(cv2.MORPH_RECT, (25, 3)))
        coords = cv2.findNonZero(binv)
        if coords is not None and len(coords) >= 80:
            ang = cv2.minAreaRect(coords)[-1]
            ang = ang + 90 if ang < -45 else (ang - 90 if ang > 45 else ang)
            if cfg.DESKEW_MIN_DEG <= abs(ang) <= cfg.DESKEW_MAX_DEG:
                a = np.asarray(img)
                h, w = a.shape[:2]
                M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, 1.0)
                cos, sin = abs(M[0, 0]), abs(M[0, 1])
                nw, nh = int(h * sin + w * cos), int(h * cos + w * sin)
                M[0, 2] += nw / 2 - w / 2
                M[1, 2] += nh / 2 - h / 2
                img = Image.fromarray(cv2.warpAffine(a, M, (nw, nh), flags=cv2.INTER_CUBIC,
                                                     borderMode=cv2.BORDER_REPLICATE))
                meta["skew_deg"] = round(float(ang), 2)
                applied.append(f"deskew_{ang:+.2f}")

    img = _resize_max(img, max_dimension)
    applied.append(f"resize_max_{max_dimension}")

    if cv2 is not None:
        if quality.get("needs_contrast_enhancement"):
            if quality.get("illumination_uniformity", 0) > cfg.ILLUMINATION_POOR:
                img = _flatten_illumination(img)
                applied.append("flatten_illumination")
            img = _clahe(img)
            applied.append("clahe")
        if quality.get("needs_denoise"):
            img = _denoise(img)
            applied.append("denoise")

    meta.update({"applied": applied, "output_size": list(img.size),
                 "approx_visual_tokens": int(img.width * img.height / 784),
                 "preprocessing_time": round(time.perf_counter() - t0, 3)})
    return img, meta


print("CELL 14 ready: analyze_page_quality(), preprocess_page()")

In [ ]:
# =========================================================================
# CELL 15 — REGION OF INTEREST EXTRACTION (shared by MRZ, names, titles)
# =========================================================================
# One mechanism serves every targeted crop: map the region back to PDF coordinates and
# RE-RENDER it from the vector source at high DPI. Upscaling a crop of an already-downscaled
# page interpolates pixels that were discarded; re-rendering recovers real detail.
def _box_to_pdf_rect(box: Tuple[int, int, int, int], view_size: Tuple[int, int],
                     pdf_rect: Tuple[float, float, float, float],
                     rotation: int) -> Optional[Tuple[float, float, float, float]]:
    """Map a box in the ORIENTED view back to PDF points. Exact only for 0/90/180/270."""
    W, H = view_size
    x0, y0, x1, y1 = box
    fx0, fy0, fx1, fy1 = x0 / W, y0 / H, x1 / W, y1 / H
    r = rotation % 360
    if r == 0:
        a, b, c, d = fx0, fy0, fx1, fy1
    elif r == 90:
        a, b, c, d = fy0, 1 - fx1, fy1, 1 - fx0
    elif r == 180:
        a, b, c, d = 1 - fx1, 1 - fy1, 1 - fx0, 1 - fy0
    elif r == 270:
        a, b, c, d = 1 - fy1, fx0, 1 - fy0, fx1
    else:
        return None
    px0, py0, px1, py1 = pdf_rect
    pw, ph = px1 - px0, py1 - py0
    return (px0 + a * pw, py0 + b * ph, px0 + c * pw, py0 + d * ph)


def extract_region(box: Tuple[int, int, int, int], view_img: Image.Image,
                   rendered: "RenderedPage", pdf_path: Path, view_meta: Dict[str, Any],
                   dpi: int, max_width: int, pad: float = 0.0,
                   cfg: Config = CFG) -> Dict[str, Any]:
    """Highest-resolution image obtainable for a region, plus where it came from."""
    W, H = view_img.size
    if pad:
        px, py = int(pad * W), int(pad * H)
        box = (max(0, box[0] - px), max(0, box[1] - py),
               min(W, box[2] + px), min(H, box[3] + py))
    box = (max(0, box[0]), max(0, box[1]), min(W, box[2]), min(H, box[3]))
    img, source = None, None

    # A deskew warp is not a rectangle mapping, so the clip path is only exact without it.
    deskewed = abs(view_meta.get("skew_deg", 0.0)) >= cfg.DESKEW_MIN_DEG
    if not deskewed and pymupdf is not None:
        rect = _box_to_pdf_rect(box, (W, H), rendered.pdf_rect, view_meta.get("rotation", 0))
        if rect:
            clip = render_pdf_clip(pdf_path, rendered.page_number, rect, dpi)
            if clip is not None and clip.width >= 150:
                rot = view_meta.get("rotation", 0) % 360
                if rot:
                    clip = clip.rotate(rot, expand=True)
                img, source = clip, "pdf_clip_rerender"
    if img is None:
        crop = view_img.crop(box)
        f = max(1.0, min(4.0, max_width / max(1, crop.width)))
        img = crop.resize((int(crop.width * f), int(crop.height * f)), Image.LANCZOS)
        source = "pixel_upscale"
    if img.width > max_width:
        s = max_width / img.width
        img = img.resize((max_width, max(1, int(img.height * s))), Image.LANCZOS)
    return {"image": img, "source": source, "bbox": list(box),
            "dimensions": [img.width, img.height],
            "approx_visual_tokens": int(img.width * img.height / 784)}


def page_tiles(view_img: Image.Image, n: int = 2) -> List[Tuple[int, int, int, int]]:
    """Horizontal tiles with a small overlap. A half page re-rendered at the same max dimension
    carries ~2x the linear resolution: this is how small print is recovered without sending an
    enormous full-page image. Layout-agnostic: no hard-coded field coordinates."""
    W, H = view_img.size
    out, step = [], H / max(1, n)
    for i in range(n):
        y0 = max(0, int(i * step - 0.06 * step))
        y1 = min(H, int((i + 1) * step + 0.06 * step))
        out.append((0, y0, W, y1))
    return out


def title_band(view_img: Image.Image, fraction: float = 0.34) -> Tuple[int, int, int, int]:
    W, H = view_img.size
    return (0, 0, W, int(H * fraction))


print("CELL 15 ready: extract_region(), page_tiles(), title_band()")

In [ ]:
# =========================================================================
# CELL 16 — MRZ CANDIDATE DETECTION  (runs on EVERY page, never assumes page 2)
# =========================================================================
# The MRZ is located SPATIALLY first, then read. Detection is texture-based, so it survives
# different passport layouts, scanners, margins and small positional shifts.
#
# Pipeline: blackhat (dark text on light ground) -> Sobel-x (MRZ glyphs are stroke-dense)
#           -> closing (characters into lines) -> closing (lines into one block) -> contours.
#
# What must NOT be accepted as an MRZ, and the measurement that rejects it:
#   photograph / portrait      -> colour saturation and edge-orientation entropy are high,
#                                 horizontal line structure is absent  (saturation, line_count)
#   Arabic identity text       -> cursive with dots above/below gives high vertical-extent
#                                 variance and a ragged right edge     (height_uniformity, fill)
#   normal printed fields      -> "NOM", "DATE DE NAISSANCE" and their values are short and
#                                 interrupted by whitespace            (width_frac, fill)
#   a document number printed
#   elsewhere on the card      -> single short run, wrong aspect ratio (aspect, width_frac)
#   borders, holograms,
#   watermarks, guilloche      -> low fill after thresholding, no repeating glyph pitch
#                                 (fill, pitch_regularity)
# A candidate must pass ALL geometric gates and then clear MRZ_MIN_SCORE. The decisive
# confirmation is structural: the transcription must look like MRZ lines (CELL 19).
@dataclass
class MRZCandidate:
    page_number: int
    bbox: Tuple[int, int, int, int]
    score: float
    n_lines: int
    width_frac: float
    rel_y: float
    fill: float
    height_uniformity: float
    pitch_regularity: float
    saturation: float
    method: str
    rejected_reason: Optional[str] = None


def _glyph_stats(roi_bin: np.ndarray) -> Tuple[float, float]:
    """(height_uniformity, pitch_regularity) of the connected components inside a band.

    MRZ is OCR-B: one fixed glyph height and one fixed pitch. Arabic, proportional print and
    decorative elements all score low on at least one of these."""
    if cv2 is None or roi_bin.size == 0:
        return 0.0, 0.0
    n, _, stats, cents = cv2.connectedComponentsWithStats(roi_bin, connectivity=8)
    hs, xs = [], []
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        if h >= 3 and area >= 4 and w <= roi_bin.shape[1] * 0.1:
            hs.append(h)
            xs.append(cents[i][0])
    if len(hs) < 8:
        return 0.0, 0.0
    hs = np.asarray(hs, dtype=np.float32)
    height_uniformity = float(np.clip(1.0 - hs.std() / (hs.mean() + 1e-6), 0, 1))
    xs = np.sort(np.asarray(xs, dtype=np.float32))
    gaps = np.diff(xs)
    gaps = gaps[(gaps > 1) & (gaps < roi_bin.shape[1] * 0.08)]
    pitch_regularity = 0.0 if gaps.size < 6 else \
        float(np.clip(1.0 - gaps.std() / (gaps.mean() + 1e-6), 0, 1))
    return round(height_uniformity, 3), round(pitch_regularity, 3)


def detect_mrz_candidates(view_img: Image.Image, page_number: int,
                          cfg: Config = CFG) -> List[MRZCandidate]:
    """Return ranked MRZ candidates for ONE page. Empty list is a legitimate answer."""
    if cv2 is None:
        return []
    W0, H0 = view_img.size
    scale = min(1.0, 1200 / W0)
    small = view_img.resize((max(1, int(W0 * scale)), max(1, int(H0 * scale))), Image.BILINEAR)
    gray = np.asarray(small.convert("L"), dtype=np.uint8)
    hsv_sat = np.asarray(small.convert("HSV"), dtype=np.uint8)[:, :, 1]
    H, W = gray.shape

    rect_k = cv2.getStructuringElement(cv2.MORPH_RECT, (max(9, W // 60), 5))
    sq_k = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    blackhat = cv2.morphologyEx(cv2.GaussianBlur(gray, (3, 3), 0), cv2.MORPH_BLACKHAT, rect_k)
    grad = np.absolute(cv2.Sobel(blackhat, cv2.CV_32F, 1, 0, ksize=-1))
    mn, mx = float(grad.min()), float(grad.max())
    grad = ((grad - mn) / (mx - mn + 1e-6) * 255).astype("uint8")
    grad = cv2.morphologyEx(grad, cv2.MORPH_CLOSE, rect_k)
    thresh = cv2.threshold(grad, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, sq_k)
    thresh = cv2.erode(thresh, None, iterations=2)
    thresh[:, :int(0.02 * W)] = 0                 # suppress page borders
    thresh[:, int(0.98 * W):] = 0

    ink = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cands: List[MRZCandidate] = []
    for c in cnts:
        x, y, w, h = cv2.boundingRect(c)
        reason = None
        ar, wf = w / float(max(1, h)), w / float(W)
        if h < 4:
            continue
        if wf < 0.45:
            reason = "too_narrow: a document number or a printed field, not a full-width MRZ"
        elif ar < 4.0:
            reason = "aspect_ratio: a block, not machine-readable lines"
        elif h > 0.30 * H:
            reason = "too_tall: a photo or a text block, not an MRZ band"
        roi_t = thresh[y:y + h, x:x + w]
        roi_ink = ink[y:y + h, x:x + w]
        fill = float((roi_t > 0).mean()) if roi_t.size else 0.0
        sat = float(hsv_sat[y:y + h, x:x + w].mean()) / 255.0 if roi_t.size else 0.0
        if reason is None and fill < 0.25:
            reason = "low_fill: decorative or security pattern, not dense glyph rows"
        if reason is None and sat > 0.35:
            reason = "colourful: portrait or security print, MRZ is monochrome"

        prof = (roi_t > 0).sum(axis=1).astype(np.float32)
        lines, run = 0, False
        for v in prof:
            hot = v > 0.35 * w
            if hot and not run:
                lines, run = lines + 1, True
            elif not hot:
                run = False
        hu, pr = _glyph_stats(roi_ink)
        if reason is None and lines > 4:
            reason = "too_many_lines: a paragraph of printed text, not an MRZ"
        if reason is None and hu < 0.45:
            reason = ("glyph heights not uniform: proportional or cursive script "
                      "(e.g. Arabic identity text), not OCR-B")

        rel_y = (y + h / 2) / H
        score = (min(ar / 20.0, 1.0) * 0.20 + wf * 0.20 + fill * 0.15 +
                 hu * 0.20 + pr * 0.15 +
                 (0.05 if lines in (2, 3) else 0.0) +
                 (0.05 if rel_y > 0.55 else 0.0))   # bottom is typical, never required
        cand = MRZCandidate(
            page_number=page_number,
            bbox=(int(x / scale), int(y / scale), int((x + w) / scale), int((y + h) / scale)),
            score=round(float(score), 3), n_lines=int(lines), width_frac=round(wf, 3),
            rel_y=round(float(rel_y), 3), fill=round(fill, 3), height_uniformity=hu,
            pitch_regularity=pr, saturation=round(sat, 3), method="morphology+glyph_stats",
            rejected_reason=reason)
        cands.append(cand)

    accepted = [c for c in cands if c.rejected_reason is None and c.score >= cfg.MRZ_MIN_SCORE]
    accepted.sort(key=lambda c: -c.score)
    return accepted[:cfg.MRZ_CANDIDATES_PER_PAGE]


def localize_mrz(all_candidates: List[MRZCandidate]) -> Optional[MRZCandidate]:
    """Choose the MRZ across ALL pages of the document.

    The MRZ may sit on page 1 (ID cards, many passports photographed as a single page) or on
    page 2 (a passport whose data page is scanned second). Selection is by evidence, not by
    page number."""
    if not all_candidates:
        return None
    return sorted(all_candidates, key=lambda c: -c.score)[0]


print("CELL 16 ready: detect_mrz_candidates() [every page], localize_mrz()")

In [ ]:
# =========================================================================
# CELL 17 — MRZ CROP, PREPROCESSING VARIANTS AND UPSCALING
# =========================================================================
# The MRZ gets its own preprocessing path, different from a normal page: the band is narrow, the
# glyphs are fixed-pitch, and horizontal resolution is what decides legibility.
def _mrz_deskew(img: Image.Image, cfg: Config = CFG) -> Tuple[Image.Image, float]:
    """Small-angle correction using the band's own text rows."""
    if cv2 is None:
        return img, 0.0
    g = np.asarray(img.convert("L"))
    binv = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    coords = cv2.findNonZero(cv2.dilate(binv, cv2.getStructuringElement(cv2.MORPH_RECT, (31, 3))))
    if coords is None or len(coords) < 60:
        return img, 0.0
    ang = cv2.minAreaRect(coords)[-1]
    ang = ang + 90 if ang < -45 else (ang - 90 if ang > 45 else ang)
    if not (0.3 <= abs(ang) <= 8.0):
        return img, 0.0
    a = np.asarray(img)
    h, w = a.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, 1.0)
    return Image.fromarray(cv2.warpAffine(a, M, (w, h), flags=cv2.INTER_CUBIC,
                                          borderMode=cv2.BORDER_REPLICATE)), round(float(ang), 2)


def preprocess_mrz(img: Image.Image, variant: str, cfg: Config = CFG
                   ) -> Tuple[Image.Image, List[str]]:
    """Named preprocessing variants. Each retry uses a DIFFERENT one -- never the same input
    twice. The original crop is always preserved by the caller for comparison."""
    ops: List[str] = []
    if cv2 is None:
        return img, ops
    out = img
    if variant in ("standard", "aggressive", "binarised"):
        out, ang = _mrz_deskew(out, cfg)
        if ang:
            ops.append(f"deskew_{ang:+.2f}")
    g = np.asarray(out.convert("L"))
    if variant == "standard":
        g = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 4)).apply(g)
        ops.append("clahe")
    elif variant == "aggressive":
        bg = cv2.GaussianBlur(g.astype(np.float32), (0, 0), sigmaX=max(g.shape) / 12.0)
        g = np.clip(g / (bg + 1e-3) * float(np.median(bg)), 0, 255).astype(np.uint8)
        ops.append("flatten_illumination")
        g = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(16, 2)).apply(g)
        ops.append("clahe_strong")
        g = cv2.bilateralFilter(g, 5, 50, 50)
        ops.append("bilateral_denoise")
    elif variant == "binarised":
        g = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 4)).apply(g)
        g = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY, 31, 11)
        ops.append("adaptive_threshold")     # last resort: binarisation costs stroke detail
    if cfg.ENABLE_SHARPENING and variant != "binarised":
        blur = cv2.GaussianBlur(g, (0, 0), 1.2)
        g = cv2.addWeighted(g, 1.5, blur, -0.5, 0)
        ops.append("unsharp")                # linear filter: raises edge contrast, invents nothing
    return Image.fromarray(g).convert("RGB"), ops


def crop_mrz(candidate: MRZCandidate, view_img: Image.Image, rendered: "RenderedPage",
             pdf_path: Path, view_meta: Dict[str, Any], variant: str = "standard",
             cfg: Config = CFG) -> Dict[str, Any]:
    """Isolate the ENTIRE MRZ band and deliver it at OCR-grade resolution."""
    region = extract_region(candidate.bbox, view_img, rendered, pdf_path, view_meta,
                            cfg.REGION_RENDER_DPI, cfg.MRZ_MAX_WIDTH, pad=0.02, cfg=cfg)
    original = region["image"]

    # MRZ-specific upscaling: unlike a page, the target is a minimum CHARACTER height.
    n_lines = max(2, candidate.n_lines or 2)
    est_char = original.height / n_lines * 0.55
    img = original
    if est_char < cfg.MRZ_MIN_CHAR_HEIGHT:
        f = min(cfg.MRZ_UPSCALE_FACTOR, cfg.MRZ_MIN_CHAR_HEIGHT / max(1e-6, est_char))
        f = min(f, cfg.MRZ_MAX_WIDTH / max(1, img.width))
        if f > 1.02:
            img = img.resize((int(img.width * f), int(img.height * f)), Image.LANCZOS)
            est_char *= f
    processed, ops = preprocess_mrz(img, variant, cfg)
    return {"original": original, "image": processed, "source": region["source"],
            "bbox": region["bbox"], "variant": variant, "ops": ops,
            "crop_width": processed.width, "crop_height": processed.height,
            "estimated_char_height_px": round(est_char, 1),
            "sufficient_resolution": bool(est_char >= cfg.MRZ_MIN_CHAR_HEIGHT),
            "approx_visual_tokens": region["approx_visual_tokens"],
            "n_lines_assumed": n_lines}


print("CELL 17 ready: crop_mrz(), preprocess_mrz() [standard|aggressive|binarised]")

In [ ]:
# =========================================================================
# CELL 18 — HANDWRITING REGION PROCESSING
# =========================================================================
# Handwriting needs stroke continuity, not contrast extremes: binarisation breaks thin pen
# strokes and is never used here.
def preprocess_handwriting(img: Image.Image, variant: str = "standard",
                           cfg: Config = CFG) -> Tuple[Image.Image, List[str]]:
    ops: List[str] = []
    if cv2 is None:
        return img, ops
    g = np.asarray(img.convert("L"))
    if variant == "standard":
        g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)
        ops.append("clahe")
    else:                                    # "contrast": form rules and shading removed harder
        bg = cv2.GaussianBlur(g.astype(np.float32), (0, 0), sigmaX=max(g.shape) / 16.0)
        g = np.clip(g / (bg + 1e-3) * float(np.median(bg)), 0, 255).astype(np.uint8)
        ops.append("flatten_illumination")
        g = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(g)
        ops.append("clahe_strong")
    if cfg.ENABLE_SHARPENING:
        blur = cv2.GaussianBlur(g, (0, 0), 1.0)
        g = cv2.addWeighted(g, 1.3, blur, -0.3, 0)
        ops.append("mild_unsharp")
    return Image.fromarray(g).convert("RGB"), ops


def handwriting_region(box: Tuple[int, int, int, int], view_img: Image.Image,
                       rendered: "RenderedPage", pdf_path: Path, view_meta: Dict[str, Any],
                       variant: str = "standard", cfg: Config = CFG) -> Dict[str, Any]:
    """High-resolution crop for a handwritten field, with its own upscale factor."""
    region = extract_region(box, view_img, rendered, pdf_path, view_meta,
                            cfg.REGION_RENDER_DPI, cfg.REGION_MAX_WIDTH, pad=0.01, cfg=cfg)
    img = region["image"]
    if img.width < cfg.REGION_MAX_WIDTH:
        f = min(cfg.HANDWRITING_UPSCALE_FACTOR, cfg.REGION_MAX_WIDTH / max(1, img.width))
        if f > 1.02:
            img = img.resize((int(img.width * f), int(img.height * f)), Image.LANCZOS)
    processed, ops = preprocess_handwriting(img, variant, cfg)
    return {"original": region["image"], "image": processed, "source": region["source"],
            "bbox": region["bbox"], "variant": variant, "ops": ops,
            "dimensions": [processed.width, processed.height],
            "approx_visual_tokens": int(processed.width * processed.height / 784)}


def save_debug(customer_id: str, document: str, page: Optional[int], task: str,
               name: str, img: Image.Image, cfg: Config = CFG) -> Optional[str]:
    """debug_images/<customer_id>/<document>/page_<n>/<task>/<name>.png"""
    if not (cfg.SAVE_DEBUG_IMAGES or cfg.SAVE_DEBUG_ON_FAILURE):
        return None
    d = DIRS["debug"] / str(customer_id) / re.sub(r"[^A-Za-z0-9_]+", "_", document) / \
        (f"page_{page}" if page else "document") / task
    d.mkdir(parents=True, exist_ok=True)
    path = d / f"{name}.png"
    try:
        img.save(path)
        return str(path)
    except Exception as exc:
        log.warning("debug image save failed: %s", exc)
        return None


print("CELL 18 ready: handwriting_region(), preprocess_handwriting(), save_debug()")

In [ ]:
# =========================================================================
# CELL 19 — QWEN FP8 INFERENCE
# =========================================================================
# Generation settings and why:
#   do_sample=False (TEMPERATURE=0) .. sampling is the mechanism by which an unseen character
#                                      gets chosen; greedy is also reproducible, which a KYC
#                                      audit trail requires.
#   repetition_penalty=1.0 ........... MUST stay 1.0. Above it, already-emitted tokens are
#                                      penalised, corrupting exactly what we transcribe: 1980,
#                                      AA1122, and the <<<<< filler of an MRZ.
#   no_repeat_ngram_size=0 ........... same reason; it would forbid "<<" outright.
#   brace-balance stop ............... ends the call when the JSON object closes instead of
#                                      decoding to the cap.
#   "{" prefill ...................... forces JSON immediately: no preamble, fewer parse errors.
#   enable_thinking=False ............ Qwen3 templates enable thinking by default; a <think>
#                                      block burns hundreds of tokens and invites reasoning
#                                      toward a plausible answer rather than a read one.
INFERENCE_LOG: List[Dict[str, Any]] = []


@dataclass
class GenResult:
    raw_model_output: str
    n_output_tokens: int
    n_input_tokens: int
    inference_time: float
    generation_time: float
    token_texts: List[str] = field(default_factory=list)
    token_logprobs: List[float] = field(default_factory=list)
    truncated: bool = False
    error: Optional[str] = None
    error_kind: Optional[str] = None
    degraded: Optional[str] = None


def is_oom_error(exc: BaseException) -> bool:
    if torch is not None and isinstance(exc, getattr(torch.cuda, "OutOfMemoryError", ())):
        return True
    t = f"{type(exc).__name__}: {exc}".lower()
    return "out of memory" in t or "outofmemory" in t


class _BraceStop:
    def __init__(self, tokenizer, prompt_len: int, open_offset: int = 1):
        self.tok, self.prompt_len, self.open_offset, self.every = \
            tokenizer, prompt_len, open_offset, 8

    def __call__(self, input_ids, scores, **kw):
        n = input_ids.shape[0]
        done = torch.zeros(n, dtype=torch.bool, device=input_ids.device)
        gen_len = input_ids.shape[1] - self.prompt_len
        if gen_len < 8 or gen_len % self.every:
            return done
        for i in range(n):
            txt = self.tok.decode(input_ids[i, self.prompt_len:], skip_special_tokens=True)
            depth, in_str, esc = self.open_offset, False, False
            for c in txt:
                if in_str:
                    if esc:        esc = False
                    elif c == "\\": esc = True
                    elif c == '"': in_str = False
                    continue
                if c == '"':   in_str = True
                elif c == "{": depth += 1
                elif c == "}":
                    depth -= 1
                    if depth <= 0:
                        done[i] = True
                        break
        return done


class _LogprobRecorder:
    """Chosen-token log-probabilities via a LogitsProcessor.

    Not output_scores=True: that retains a [batch, vocab] tensor per step for the whole
    generation (~100 MB per sequence at a 150k vocab), held when memory is tightest. Under
    greedy decoding the chosen token is the argmax, so one scalar per step is equivalent. This
    is an OBJECTIVE signal from the decoder, unlike a confidence the model simply writes out."""

    def __init__(self, batch_size: int):
        self.token_ids: List[List[int]] = [[] for _ in range(batch_size)]
        self.logprobs: List[List[float]] = [[] for _ in range(batch_size)]

    def __call__(self, input_ids, scores):
        lp = torch.log_softmax(scores.float(), dim=-1)
        top = lp.argmax(dim=-1)
        vals = lp.gather(1, top.unsqueeze(1)).squeeze(1)
        for i, (t, v) in enumerate(zip(top.tolist(), vals.tolist())):
            if i < len(self.token_ids):
                self.token_ids[i].append(int(t))
                self.logprobs[i].append(float(v))
        return scores


OCR_SYSTEM = """You are a visual transcription engine. The supplied image is the only source of truth.

Read only what is visually present. Do not guess, infer, reconstruct, correct, translate, transliterate, or complete unreadable characters. Never use external knowledge, database information, or other documents.
When a value is requested in Latin letters, transcribe only Latin characters actually printed or written on this image.
Mark any single character you cannot read reliably with ?. If you cannot read a value at all, return null for it.
Output JSON only. No markdown, no commentary, no reasoning."""


def run_qwen_ocr(image: Image.Image, user_prompt: str, max_new_tokens: int,
                 engine=None, system: str = OCR_SYSTEM, cfg: Config = CFG,
                 task: str = "generic", customer_id: str = "", document: str = "",
                 page: Optional[int] = None, retry_number: int = 0,
                 _depth: int = 0) -> GenResult:
    """One inference call. Model and processor come from the singleton: nothing is initialised
    here, per customer, per document, per page or per retry."""
    engine = engine or ENGINE
    t_all = time.perf_counter()

    if isinstance(engine, MockEngine):
        return GenResult(raw_model_output="{}", n_output_tokens=0, n_input_tokens=0,
                         inference_time=0.01, generation_time=0.01)

    from transformers import StoppingCriteriaList, LogitsProcessorList
    messages = [{"role": "system", "content": [{"type": "text", "text": system}]},
                {"role": "user", "content": [{"type": "image"},
                                             {"type": "text", "text": user_prompt}]}]
    try:
        text = engine.processor.apply_chat_template(messages, tokenize=False,
                                                    add_generation_prompt=True,
                                                    enable_thinking=not cfg.DISABLE_THINKING)
    except TypeError:
        text = engine.processor.apply_chat_template(messages, tokenize=False,
                                                    add_generation_prompt=True)
    text += "{"

    try:
        inputs = engine.processor(text=[text], images=[image], padding=True,
                                  return_tensors="pt").to(engine.model.device)
        prompt_len = int(inputs["input_ids"].shape[1])
        rec = _LogprobRecorder(1)
        t_gen = time.perf_counter()
        with torch.inference_mode():
            out = engine.model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, num_beams=1,
                repetition_penalty=cfg.REPETITION_PENALTY,
                no_repeat_ngram_size=cfg.NO_REPEAT_NGRAM_SIZE,
                stopping_criteria=StoppingCriteriaList(
                    [_BraceStop(engine.tokenizer, prompt_len, 1)]),
                logits_processor=LogitsProcessorList([rec]),
                return_dict_in_generate=True, output_scores=False,
                pad_token_id=getattr(engine.tokenizer, "pad_token_id", None)
                             or getattr(engine.tokenizer, "eos_token_id", None))
        gen_s = time.perf_counter() - t_gen
        ids = out.sequences[0, prompt_len:].detach().to("cpu")
        del out, inputs                    # drop GPU refs; no empty_cache() per page
        n = int(ids.shape[0])
        result = GenResult(
            raw_model_output="{" + engine.tokenizer.decode(ids, skip_special_tokens=True),
            n_output_tokens=n, n_input_tokens=prompt_len,
            inference_time=round(time.perf_counter() - t_all, 3),
            generation_time=round(gen_s, 3),
            token_texts=[engine.tokenizer.decode([t]) for t in rec.token_ids[0][:n]],
            token_logprobs=rec.logprobs[0][:n], truncated=bool(n >= max_new_tokens))
        engine.consecutive_ooms = 0
    except Exception as exc:
        if not is_oom_error(exc):
            result = GenResult(raw_model_output="", n_output_tokens=0, n_input_tokens=0,
                               inference_time=round(time.perf_counter() - t_all, 3),
                               generation_time=0.0,
                               error=f"{type(exc).__name__}: {exc}", error_kind="inference")
        else:
            release_cuda_cache()           # the one place empty_cache() is justified
            engine.consecutive_ooms += 1
            if _depth < cfg.OOM_MAX_DOWNSCALES:
                f = cfg.OOM_DOWNSCALE_FACTOR
                small = image.resize((max(64, int(image.width * f)),
                                      max(64, int(image.height * f))), Image.LANCZOS)
                log.warning("OOM; retrying this image at %d%%", int(f * 100))
                r = run_qwen_ocr(small, user_prompt, max_new_tokens, engine, system, cfg,
                                 task, customer_id, document, page, retry_number, _depth + 1)
                r.degraded = f"oom_downscale_x{f ** (_depth + 1):.2f}"
                return r
            result = GenResult(raw_model_output="", n_output_tokens=0, n_input_tokens=0,
                               inference_time=round(time.perf_counter() - t_all, 3),
                               generation_time=0.0,
                               error=f"{type(exc).__name__}: {exc}", error_kind="oom")

    INFERENCE_LOG.append({
        "customer_id": customer_id, "document": document, "page": page, "task": task,
        "image_width": image.width, "image_height": image.height,
        "input_tokens": result.n_input_tokens, "output_tokens": result.n_output_tokens,
        "inference_time": result.inference_time, "generation_time": result.generation_time,
        "retry_number": retry_number, "truncated": result.truncated,
        "error": result.error_kind, "timestamp": utcnow()})
    return result


print("CELL 19 ready: run_qwen_ocr()")

In [ ]:
# =========================================================================
# CELL 20 — TASK-SPECIFIC PROMPTS
# =========================================================================
# No null-filled template is ever shown to the model. The previous pipeline specified the output
# shape as {"value": null, "confidence": "unreadable"} for every field, which made copying the
# failure answer a valid, zero-risk completion -- a large part of why it returned mostly null.
# Here each prompt lists the keys and shows a FILLED worked example.
LINES_PROMPT = """Transcribe every line of text you can see in this image, in reading order.

Return exactly:
{"lines": ["...", "..."], "scripts": ["latin"], "has_handwriting": false}

Rules: copy each line exactly as printed or written, including labels. Use ? for any single character you cannot read. Do not translate, transliterate, reorder or summarise. Include Arabic or Cyrillic lines in their own script. Return only the JSON object."""


def identity_fields_prompt(fields: Optional[Sequence[str]] = None) -> str:
    fields = list(fields or [f for f in IDENTITY_FIELDS if f != "mrz"])
    keys = ", ".join(f'"{f}"' for f in fields)
    return f"""Read this identity document image and report these fields: {keys}.

Return one JSON object whose keys are exactly those names. Each maps to an object with "value" and "confidence", where confidence is high, medium, low or unreadable and describes how clearly you can SEE the characters.

Worked example of the shape (your values must come from THIS image):
{{"surname_latin": {{"value": "BEN ALI", "confidence": "high"}},
 "date_of_birth": {{"value": "01.01.1980", "confidence": "high"}},
 "document_number": {{"value": "12AB4?678", "confidence": "low"}}}}

"surname_latin" and "given_names_latin" must contain only Latin letters actually printed on the document; if the document shows an Arabic name, do not transliterate it. A field that is not on this page gets value null. Return only the JSON object."""


MRZ_PROMPT = """This image is the machine readable zone (MRZ) of an identity document, cropped and enlarged. Read ONLY this region.

Transcribe the MRZ lines exactly as printed: preserve every < character, the character order and the line order. Insert no spaces. Do not repair, translate or transliterate anything. Check digits must NOT be used to decide an unclear character. Mark a character you cannot read with ?.

Return exactly:
{"mrz": {"value": "P<UTOERIKSSON<<ANNA<MARIA<<<<<<<<<<<<<<<<<<<\\nL898902C36UTO7408122F1204159ZE184226B<<<<<10", "confidence": "high"}}

with the lines of THIS image, separated by \\n. If you cannot read the region at all, use null. Return only the JSON object."""


TITLE_PROMPT = """Transcribe the printed heading text at the top of this document image.

Return exactly:
{"title_text": {"value": "CONVENTION DE COMPTE", "confidence": "high"}, "other_headings": {"value": "Agence de Paris", "confidence": "medium"}}

with the text of THIS image. Copy it exactly as written, including spelling that looks unusual. Do not summarise, translate or correct. If nothing is readable use null. Return only the JSON object."""


def name_prompt(handwritten: bool = False) -> str:
    hw = ("\nThe name is HANDWRITTEN. Read the strokes directly, letter by letter. Mark any "
          "letter you cannot read with ?. Never complete a name from context.") if handwritten \
        else ""
    return f"""Read the client's surname and given names from this document image.{hw}

Return exactly:
{{"surname_latin": {{"value": "BEN ALI", "confidence": "high", "text_type": "handwritten"}},
 "given_names_latin": {{"value": "MOHAMED", "confidence": "medium", "text_type": "handwritten"}}}}

with the values from THIS image. Transcribe only Latin letters actually visible here. Do not transliterate Arabic or Cyrillic, do not correct spelling, do not use any other document. "text_type" is printed, handwritten or mixed. Use null for a value you cannot read. Return only the JSON object."""


print("CELL 20 ready: prompts (no null template is ever shown to the model)")

In [ ]:
# =========================================================================
# CELL 21 — JSON PARSING  (raw -> parsed -> validated, kept separate)
# =========================================================================
CONF_BANDS = ("high", "medium", "low", "unreadable")
CONF_RANK = {"high": 3, "medium": 2, "low": 1, "unreadable": 0}
CONF_FLOAT = {"high": 0.92, "medium": 0.7, "low": 0.4, "unreadable": 0.0}
NULLISH = {"", "null", "none", "n/a", "na", "unreadable", "not visible", "-", "--"}


def _first_json_object(text: str) -> Optional[str]:
    if not text:
        return None
    t = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    t = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", t.strip())
    start = t.find("{")
    if start < 0:
        return None
    depth, in_str, esc = 0, False, False
    for i in range(start, len(t)):
        c = t[i]
        if in_str:
            if esc:        esc = False
            elif c == "\\": esc = True
            elif c == '"': in_str = False
            continue
        if c == '"':   in_str = True
        elif c == "{": depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                return t[start:i + 1]
    return t[start:] + "}" * depth if depth > 0 else None      # tolerate a truncated tail


def _clean(v: Any, name: str = "") -> Optional[str]:
    if isinstance(v, (list, tuple)):
        v = ("\n" if name == "mrz" else " ").join(str(x) for x in v if x is not None)
    if v is None or isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        v = str(v)
    if not isinstance(v, str):
        return None
    s = v.strip()
    return None if s.lower() in NULLISH else s


def parse_json_response(raw: str, fields: Sequence[str]) -> Tuple[Dict[str, Any], bool, Optional[str]]:
    """Coerce model output onto the requested keys. SUBTRACTIVE ONLY: a missing key becomes null,
    an unknown confidence is downgraded (never promoted), a null value is forced to 'unreadable'.
    Unparsable output fails CLOSED. The raw string is kept by the caller, untouched."""
    fields = list(fields)
    block = _first_json_object(raw)
    empty = {f: {"value": None, "confidence": "unreadable"} for f in fields}
    if block is None:
        return empty, False, "no_json_in_output"
    try:
        obj = json.loads(block)
    except json.JSONDecodeError as exc:
        try:
            obj = json.loads(re.sub(r",\s*([}\]])", r"\1", block))
        except Exception:
            return empty, False, f"json_decode_error: {exc}"
    if not isinstance(obj, dict):
        return empty, False, "not_an_object"

    lower = {str(k).strip().lower(): k for k in obj}
    out: Dict[str, Any] = {}
    for f in fields:
        key = lower.get(f) or (lower.get("machine_readable_zone") if f == "mrz" else None)
        node = obj.get(key) if key else None
        if isinstance(node, dict):
            value = _clean(node.get("value"), f)
            conf = str(node.get("confidence", "")).strip().lower()
            ttype = (str(node.get("text_type", "") or "").strip().lower() or None)
        else:
            value, conf, ttype = _clean(node, f), "", None
        if conf not in CONF_BANDS:
            conf = "low" if value is not None else "unreadable"
        if value is None:
            conf = "unreadable"
        entry: Dict[str, Any] = {"value": value, "confidence": conf}
        if ttype in ("printed", "handwritten", "mixed"):
            entry["text_type"] = ttype
        out[f] = entry
    return out, True, None


def parse_lines_response(raw: str) -> Tuple[List[str], Dict[str, Any], bool]:
    """Parse the line-transcription pass."""
    block = _first_json_object(raw)
    if block is None:
        return [], {}, False
    try:
        obj = json.loads(block)
    except Exception:
        try:
            obj = json.loads(re.sub(r",\s*([}\]])", r"\1", block))
        except Exception:
            return [], {}, False
    if not isinstance(obj, dict):
        return [], {}, False
    lines = obj.get("lines") or obj.get("text") or []
    if isinstance(lines, str):
        lines = lines.splitlines()
    lines = [str(l).strip() for l in lines if str(l).strip()]
    meta = {"scripts": obj.get("scripts"), "has_handwriting": obj.get("has_handwriting")}
    return lines, meta, True


print("CELL 21 ready: parse_json_response(), parse_lines_response()")

In [ ]:
# =========================================================================
# CELL 22 — FIELD MAPPING FROM TRANSCRIBED LINES + VALIDATION
# =========================================================================
# Mapping labelled lines to fields is deterministic and auditable. Agreement between this
# (from PASS A) and the model's own schema answer (PASS B) is an OBJECTIVE consistency signal --
# two independent reads of the same pixels -- rather than a self-reported confidence.
FIELD_LABELS: Dict[str, List[str]] = {
    "surname_latin": ["NOM", "SURNAME", "NOM DE FAMILLE", "FAMILY NAME", "APELLIDOS", "LAST NAME"],
    "given_names_latin": ["PRENOM", "PRENOMS", "GIVEN NAME", "GIVEN NAMES", "FORENAMES",
                          "FIRST NAME", "NOMBRE"],
    "date_of_birth": ["DATE DE NAISSANCE", "DATE OF BIRTH", "NE LE", "NEE LE", "GEBURTSDATUM",
                      "BIRTH DATE", "DOB"],
    "place_of_birth": ["LIEU DE NAISSANCE", "PLACE OF BIRTH", "BIRTH PLACE"],
    "nationality": ["NATIONALITE", "NATIONALITY", "NACIONALIDAD"],
    "sex": ["SEXE", "SEX", "GENDER"],
    "document_number": ["NUMERO DU DOCUMENT", "DOCUMENT NO", "PASSPORT NO", "NO DU PASSEPORT",
                        "NUMERO", "DOCUMENT NUMBER", "CARTE NO", "NO"],
    "issue_date": ["DATE DE DELIVRANCE", "DATE OF ISSUE", "DELIVRE LE", "ISSUED ON"],
    "expiry_date": ["DATE D EXPIRATION", "DATE OF EXPIRY", "EXPIRE LE", "VALABLE JUSQU AU",
                    "DATE D EXPIRATION DU DOCUMENT", "EXPIRY", "VALID UNTIL"],
    "issuing_authority": ["AUTORITE", "ISSUING AUTHORITY", "DELIVRE PAR", "AUTHORITY"],
    "personal_number": ["NUMERO PERSONNEL", "PERSONAL NO", "PERSONAL NUMBER", "CIN", "NIN",
                        "IDENTITY NUMBER"],
}
LABEL_LOOKUP = sorted(((lab, f) for f, labs in FIELD_LABELS.items() for lab in labs),
                      key=lambda x: -len(x[0]))
MRZ_LINE_RX = re.compile(r"^[A-Z0-9<]{25,50}$")


def _norm_label(s: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9 ]+", " ", strip_accents(str(s)).upper())).strip()


def map_lines_to_fields(lines: Sequence[str]) -> Dict[str, Dict[str, Any]]:
    """Label-driven mapping. Handles 'Label: value' and a label whose value is on the next line."""
    out: Dict[str, Dict[str, Any]] = {}
    norm = [_norm_label(l) for l in lines]
    for i, (raw_line, nline) in enumerate(zip(lines, norm)):
        if not nline:
            continue
        for label, fieldname in LABEL_LOOKUP:
            if fieldname in out:
                continue
            if not nline.startswith(label):
                continue
            rest = raw_line
            m = re.search(r"[:\-]\s*", raw_line)
            if m:
                rest = raw_line[m.end():]
            else:
                rest = raw_line[len(label):] if len(raw_line) > len(label) else ""
            value = rest.strip(" .:-\t")
            if not value and i + 1 < len(lines):      # value printed on the following line
                nxt = lines[i + 1].strip()
                if nxt and not any(_norm_label(nxt).startswith(l) for l, _ in LABEL_LOOKUP):
                    value = nxt
            if value:
                out[fieldname] = {"value": value, "source_line": raw_line, "label": label}
            break
    mrz_lines = [l.strip().replace(" ", "") for l in lines
                 if MRZ_LINE_RX.match(l.strip().replace(" ", "")) and
                 l.count("<") >= 3]
    if len(mrz_lines) >= 2:
        out["mrz"] = {"value": "\n".join(mrz_lines[:3]), "source_line": "mrz_lines",
                      "label": "MRZ"}
    return out


# ------------------------------- validation --------------------------------
DATE_RX = [(re.compile(r"^(\d{2})[./\- ](\d{2})[./\- ](\d{4})$"), ("d", "m", "y")),
           (re.compile(r"^(\d{4})[./\- ](\d{2})[./\- ](\d{2})$"), ("y", "m", "d")),
           (re.compile(r"^(\d{2})[./\- ](\d{2})[./\- ](\d{2})$"), ("d", "m", "yy")),
           (re.compile(r"^(\d{2})\s?([A-Z]{3})\s?(\d{2,4})$"), ("d", "mon", "y"))]
MONTHS3 = {m: i + 1 for i, m in enumerate(["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL",
                                           "AUG", "SEP", "OCT", "NOV", "DEC"])}
MRZ_SPECS = {"TD1": (3, 30), "TD2": (2, 36), "TD3": (2, 44)}


def parse_date(value: Optional[str]) -> Optional[date]:
    """For COMPARISON only. The stored transcription is never replaced by this."""
    if not value:
        return None
    s = strip_accents(str(value)).strip().upper()
    for rx, order in DATE_RX:
        m = rx.match(s)
        if not m:
            continue
        g = dict(zip(order, m.groups()))
        try:
            if "y" in g:
                y = int(g["y"])
                y = y if y > 99 else (2000 + y if y < 30 else 1900 + y)
            else:
                yy = int(g["yy"])
                y = 2000 + yy if yy < 30 else 1900 + yy
            mth = MONTHS3.get(g.get("mon"), 0) or int(g.get("m", 0))
            return date(y, mth, int(g["d"]))
        except Exception:
            return None
    return None


def mrz_check_digit(s: str) -> Optional[str]:
    w, total = (7, 3, 1), 0
    for i, c in enumerate(s):
        if c == "<":      v = 0
        elif c.isdigit(): v = int(c)
        elif c.isalpha(): v = ord(c.upper()) - 55
        else:             return None
        total += v * w[i % 3]
    return str(total % 10)


def validate_mrz(value: Optional[str]) -> Dict[str, Any]:
    """Structure, charset, length and ICAO check digits. DIAGNOSTIC ONLY.

    A failing check digit proves a character was misread; it does not reveal which one or what it
    should be. Using it to choose a replacement would be hallucination wearing a checksum."""
    r: Dict[str, Any] = {"status": "absent", "format": None, "line_lengths": [], "checks": {},
                         "flags": [], "note": "diagnostic only; never modifies the OCR result"}
    if not value:
        return r
    lines = [l.strip().replace(" ", "") for l in str(value).splitlines() if l.strip()]
    r["line_lengths"] = [len(l) for l in lines]
    if any("?" in l for l in lines):
        r["flags"].append("contains_unreadable_characters")
    fmt = next((k for k, (n, ln) in MRZ_SPECS.items()
                if len(lines) == n and all(abs(len(l) - ln) <= 2 for l in lines)), None)
    r["format"] = fmt
    if fmt is None:
        r["status"] = "structure_invalid"
        r["flags"].append(f"unexpected_layout:{len(lines)}x{r['line_lengths']}")
        return r
    if any(re.search(r"[^A-Z0-9<?]", l) for l in lines):
        r["flags"].append("unexpected_characters")
    try:
        if fmt in ("TD2", "TD3"):
            l2 = lines[1]
            for name, (a, b, cd) in {"document_number": (0, 9, 9), "date_of_birth": (13, 19, 19),
                                     "expiry_date": (21, 27, 27)}.items():
                if len(l2) > cd:
                    r["checks"][name] = {"computed": mrz_check_digit(l2[a:b]), "printed": l2[cd]}
        else:
            r["checks"]["document_number"] = {"computed": mrz_check_digit(lines[0][5:14]),
                                              "printed": lines[0][14]}
            r["checks"]["date_of_birth"] = {"computed": mrz_check_digit(lines[1][0:6]),
                                            "printed": lines[1][6]}
            r["checks"]["expiry_date"] = {"computed": mrz_check_digit(lines[1][8:14]),
                                          "printed": lines[1][14]}
    except Exception as exc:
        r["flags"].append(f"checksum_error:{exc}")
    verdicts = [c["computed"] == c["printed"] for c in r["checks"].values()
                if c.get("computed") is not None]
    r["status"] = ("not_verifiable" if not verdicts else
                   "valid" if all(verdicts) else "checksum_mismatch")
    if r["status"] == "checksum_mismatch":
        r["flags"].append("checksum_mismatch:" + ",".join(
            k for k, c in r["checks"].items() if c.get("computed") != c.get("printed")))
    return r


def looks_like_mrz(value: Optional[str]) -> bool:
    """Structural confirmation that a CANDIDATE region really was an MRZ.

    This is the decisive discriminator against Arabic text, printed fields, photos and
    decoration: whatever the morphology scored, the transcription must look like MRZ lines."""
    if not value:
        return False
    lines = [l.strip().replace(" ", "") for l in str(value).splitlines() if l.strip()]
    if len(lines) < 2:
        return False
    ok = [bool(MRZ_LINE_RX.match(l)) for l in lines[:3]]
    fillers = sum(l.count("<") for l in lines)
    return sum(ok) >= 2 and fillers >= 3


def validate_field(name: str, value: Optional[str]) -> Dict[str, Any]:
    """Format checks. Flags only: no validator may change a character."""
    e: Dict[str, Any] = {"format_valid": None, "flags": []}
    if not value:
        return e
    if "?" in str(value):
        e["flags"].append("contains_unreadable_characters")
    if name in ("date_of_birth", "issue_date", "expiry_date"):
        d = parse_date(value)
        e["format_valid"] = d is not None
        if d is None:
            e["flags"].append("unrecognised_date_format")
        else:
            e["parsed_date"] = d.isoformat()
            if name == "date_of_birth":
                age = (date.today() - d).days / 365.25
                if not 0 <= age <= 120:
                    e["format_valid"] = False
                    e["flags"].append(f"implausible_age:{age:.0f}")
    elif name in ("document_number", "personal_number"):
        core = str(value).upper().replace(" ", "")
        e["format_valid"] = bool(re.fullmatch(r"[A-Z0-9<?\-/]{4,20}", core))
        risky = sorted(set(core) & set("O0I1L5S8B2ZG6"))
        if risky and re.search(r"\d", core) and re.search(r"[A-Z]", core):
            # FLAG ONLY. Replacing O with 0 here would be the pipeline guessing.
            e["flags"].append("possible_ocr_confusion:" + "".join(risky))
    elif name.endswith("_latin"):
        e["format_valid"] = bool(re.fullmatch(r"[A-Za-z '?\-\.]+", str(value)))
        if not e["format_valid"]:
            e["flags"].append("non_latin_characters_in_latin_field")
    elif name == "sex":
        e["format_valid"] = str(value).strip().upper() in {"M", "F", "X", "H", "MALE", "FEMALE",
                                                           "MASCULIN", "FEMININ"}
    return e


print("CELL 22 ready: map_lines_to_fields(), validate_mrz(), looks_like_mrz()")

In [ ]:
# =========================================================================
# CELL 23 — CONFIDENCE AND FIELD RECORDS  (visual evidence only)
# =========================================================================
# Confidence is COMPUTED from measurable evidence, never taken from the model's word alone, and
# never from database or cross-document agreement -- those are verification, not evidence.
CONFIDENCE_WEIGHTS = {
    "pass_agreement": 0.30,     # PASS A lines vs PASS B schema: two independent reads
    "visual_clarity": 0.25,     # measured page/region quality
    "character_legibility": 0.20,  # absence of '?' markers in the transcription
    "format_validity": 0.15,    # deterministic format check
    "region_resolution": 0.10,  # glyph height actually delivered to the model
}


def compute_confidence(value: Optional[str], model_band: str, agreement: Optional[float],
                       visual_clarity: float, format_valid: Optional[bool],
                       text_height: Optional[float], cfg: Config = CFG) -> Dict[str, Any]:
    """Returns {"confidence": float, "confidence_band": str, "components": {...}, "caps": [...]}"""
    if not value:
        return {"confidence": 0.0, "confidence_band": "unreadable", "components": {},
                "caps_applied": ["no_value"]}
    comps: Dict[str, Optional[float]] = {
        "pass_agreement": agreement,
        "visual_clarity": float(np.clip(visual_clarity, 0, 1)),
        "character_legibility": 0.0 if "?" in str(value) else 1.0,
        "format_validity": None if format_valid is None else (1.0 if format_valid else 0.0),
        "region_resolution": None if not text_height else
        float(np.clip(text_height / (1.5 * cfg.MIN_TEXT_HEIGHT), 0, 1)),
    }
    usable = {k: v for k, v in comps.items() if v is not None}
    total_w = sum(CONFIDENCE_WEIGHTS[k] for k in usable) or 1.0
    score = sum(CONFIDENCE_WEIGHTS[k] * v for k, v in usable.items()) / total_w

    caps: List[str] = []
    def cap(limit, why):
        nonlocal score
        if score > limit:
            score, _ = limit, caps.append(why)
    if "?" in str(value):
        cap(0.40, "unreadable_characters_present")
    if format_valid is False:
        cap(0.50, "format_invalid")
    if model_band == "low":
        cap(0.60, "model_reported_low")
    if model_band == "unreadable":
        cap(0.35, "model_reported_unreadable")
    band = ("high" if score >= 0.80 else "medium" if score >= 0.60
            else "low" if score >= 0.35 else "unreadable")
    return {"confidence": round(float(score), 3), "confidence_band": band,
            "components": comps, "caps_applied": caps}


def make_field(name: str, value: Optional[str], model_band: str, *, document: str, page: Optional[int],
               region: str, ocr_task: str, preprocessing_variant: Optional[str] = None,
               retry_number: int = 0, agreement: Optional[float] = None,
               visual_clarity: float = 0.5, text_height: Optional[float] = None,
               text_type: Optional[str] = None, raw_output: Optional[str] = None,
               cfg: Config = CFG) -> Dict[str, Any]:
    """One auditable field record: value + confidence + full provenance."""
    val = validate_field(name, value)
    conf = compute_confidence(value, model_band, agreement, visual_clarity,
                              val.get("format_valid"), text_height, cfg)
    return {"value": value,
            "confidence": conf["confidence"], "confidence_band": conf["confidence_band"],
            "confidence_components": conf["components"], "caps_applied": conf["caps_applied"],
            "model_reported_confidence": model_band,
            "source_document": document, "source_page": page, "source_region": region,
            "ocr_task": ocr_task, "preprocessing_variant": preprocessing_variant,
            "retry_number": retry_number, "text_type": text_type,
            "validation": val, "raw_model_output": (raw_output or "")[:800],
            "match_status": None}       # filled during verification; never alters "value"


def empty_field(name: str, document: str, reason: str = "not_extracted") -> Dict[str, Any]:
    return {"value": None, "confidence": 0.0, "confidence_band": "unreadable",
            "confidence_components": {}, "caps_applied": [reason],
            "model_reported_confidence": "unreadable", "source_document": document,
            "source_page": None, "source_region": None, "ocr_task": None,
            "preprocessing_variant": None, "retry_number": 0, "text_type": None,
            "validation": {"format_valid": None, "flags": []}, "raw_model_output": "",
            "match_status": None, "failure_reason": reason}


def normalize_for_comparison(value: Optional[str]) -> Optional[str]:
    """Comparison key ONLY. Never stored in place of a raw value."""
    if not value:
        return None
    s = re.sub(r"[^A-Z ]+", " ", strip_accents(str(value)).upper())
    return re.sub(r"\s+", " ", s).strip() or None


def agreement_score(a: Optional[str], b: Optional[str]) -> Optional[float]:
    """Agreement between two independent reads of the same pixels."""
    na, nb = normalize_for_comparison(a), normalize_for_comparison(b)
    if na is None or nb is None:
        return None
    if na == nb:
        return 1.0
    return round(difflib.SequenceMatcher(None, na, nb).ratio(), 3)


print("CELL 23 ready: compute_confidence(), make_field()")

In [ ]:
# =========================================================================
# CELL 24 — DOCUMENT CLASSIFICATION / TITLE VERIFICATION
# =========================================================================
# The model TRANSCRIBES the heading; Python decides MATCH / MISMATCH / UNCERTAIN.
# Asking a VLM "is this the signature card?" invites agreement -- a leading question gets a
# leading answer. Transcribe-then-compare keeps the verdict deterministic and auditable.
#
# Note: section 26 gives the expected title as "SPICIMEN DE SIGNATURE", which looks like a typo
# for "SPECIMEN". Both spellings are accepted and the title actually seen is recorded verbatim,
# so the discrepancy stays visible instead of being normalised away.
EXPECTED_TITLES: Dict[str, Dict[str, Any]] = {
    IDENTITY_DOC: {"any_of": ["PASSEPORT", "PASSPORT", "CARTE NATIONALE D IDENTITE",
                              "CARTE D IDENTITE", "TITRE DE SEJOUR", "CARTE DE SEJOUR",
                              "PERMIS DE CONDUIRE", "REPUBLIQUE", "IDENTITY CARD"],
                   "label": "identity document"},
    DOMICILE_DOC: {"any_of": ["FACTURE", "QUITTANCE", "ATTESTATION DE DOMICILE",
                              "JUSTIFICATIF DE DOMICILE", "ELECTRICITE", "RELEVE",
                              "CONTRAT DE BAIL", "AVIS D ECHEANCE", "INVOICE"],
                   "label": "proof of address"},
    CONVENTION_DOC: {"any_of": ["CONVENTION DE COMPTE", "CONVENTION COMPTE",
                                "OUVERTURE DE COMPTE", "CONDITIONS GENERALES"],
                     "label": "account convention"},
    SIGNATURE_DOC: {"any_of": ["SPICIMEN DE SIGNATURE", "SPECIMEN DE SIGNATURE",
                               "SPECIMEN SIGNATURE"],
                    "label": "SPICIMEN DE SIGNATURE"},
}
FATCA_COMPONENTS = {
    # The contiguous phrase matters: "FATCA" plus "IDENTIFICATION" as loose tokens also occurs in
    # component 2's title, which would make component 1 swallow component 2's page.
    "component_1": {"all_of": ["FATCA IDENTIFICATION FORMS"], "expected_pages": 2,
                    "label": "FATCA Identification forms(v1.0)"},
    # "<<US PERSON>>" is treated as a wildcard: the anchor phrases must both be present.
    "component_2": {"all_of": ["FORMULAIRE D IDENTIFICATION", "AU REGARD DE LA LOI FATCA"],
                    "any_of": ["SOUSCRIPTEUR", "ASSURE", "US PERSON"], "expected_pages": 1,
                    "label": "FORMULAIRE D'IDENTIFICATION <<US PERSON>> AU REGARD DE LA LOI "
                             "FATCA SOUSCRIPTEUR/ASSURE"},
}


def normalize_title(value: Optional[str]) -> str:
    if not value:
        return ""
    s = re.sub(r"[^A-Z0-9 ]+", " ", strip_accents(str(value)).upper())
    return re.sub(r"\s+", " ", s).strip()


def read_page_title(rendered: "RenderedPage", view_img: Image.Image, pdf_path: Path,
                    view_meta: Dict[str, Any], engine, timings: Dict[str, float],
                    customer_id: str, document: str, cfg: Config = CFG) -> Dict[str, Any]:
    """Transcribe the heading band, re-rendered at high DPI."""
    t0 = time.perf_counter()
    region = extract_region(title_band(view_img), view_img, rendered, pdf_path, view_meta,
                            cfg.REGION_RENDER_DPI, cfg.REGION_MAX_WIDTH, cfg=cfg)
    timings["roi_detection"] += time.perf_counter() - t0
    gen = run_qwen_ocr(region["image"], TITLE_PROMPT, cfg.MAX_NEW_TOKENS_TITLE, engine, cfg=cfg,
                       task="title", customer_id=customer_id, document=document,
                       page=rendered.page_number)
    timings["inference"] += gen.inference_time
    parsed, ok, warn = parse_json_response(gen.raw_model_output, ["title_text", "other_headings"])
    if cfg.SAVE_DEBUG_IMAGES:
        save_debug(customer_id, document, rendered.page_number, "title", "title_band",
                   region["image"], cfg)
    return {"title_text": parsed["title_text"]["value"],
            "other_headings": parsed["other_headings"]["value"],
            "confidence": parsed["title_text"]["confidence"],
            "region_source": region["source"], "dimensions": region["dimensions"],
            "inference_time": gen.inference_time, "parse_ok": ok, "parse_warning": warn,
            "raw_model_output": gen.raw_model_output[:800], "error_kind": gen.error_kind}


def classify_title(observed: Optional[str], other: Optional[str], spec: Dict[str, Any],
                   cfg: Config = CFG) -> Dict[str, Any]:
    """Deterministic comparison. UNCERTAIN when nothing was readable -- an unreadable title is
    not a wrong document. MISMATCH only when a title WAS read and does not correspond."""
    hay = normalize_title(" | ".join([x for x in (observed, other) if x]))
    if not hay:
        return {"status": "UNCERTAIN", "observed_title": observed, "score": None,
                "reason": "title not readable"}
    missing = [p for p in spec.get("all_of", []) if normalize_title(p) not in hay]
    hits, scores = [], []
    for phrase in spec.get("any_of", []) or spec.get("all_of", []):
        n = normalize_title(phrase)
        if n and n in hay:
            hits.append(phrase)
            scores.append(1.0)
        else:
            best = max((difflib.SequenceMatcher(None, n, hay[i:i + len(n)]).ratio()
                        for i in range(0, max(1, len(hay) - len(n) + 1), 4)), default=0.0)
            scores.append(best)
    score = round(max(scores) if scores else 0.0, 3)
    if missing:
        return {"status": "MISMATCH", "observed_title": observed, "score": score,
                "threshold": cfg.TITLE_MATCH_THRESHOLD,
                "reason": "missing required phrase: " + "; ".join(missing)}
    if hits or score >= cfg.TITLE_MATCH_THRESHOLD:
        return {"status": "MATCH", "observed_title": observed, "score": score,
                "matched_phrase": hits[0] if hits else None,
                "threshold": cfg.TITLE_MATCH_THRESHOLD,
                "reason": f"phrase found: {hits[0]}" if hits else f"fuzzy score {score}"}
    return {"status": "MISMATCH", "observed_title": observed, "score": score,
            "threshold": cfg.TITLE_MATCH_THRESHOLD,
            "reason": f"no expected phrase (best {score} < {cfg.TITLE_MATCH_THRESHOLD})"}


print("CELL 24 ready: read_page_title(), classify_title()")

In [ ]:
# =========================================================================
# CELL 25 — IDENTITY EXTRACTION  (every page; MRZ searched on every page)
# =========================================================================
def prepare_view(rendered: "RenderedPage", timings: Dict[str, float],
                 cfg: Config = CFG) -> Tuple[Image.Image, Dict[str, Any], Dict[str, Any]]:
    """Quality -> orientation/deskew/photometry -> the view the model actually sees.
    The full-resolution render in `rendered.image` is left untouched for ROI re-rendering."""
    t0 = time.perf_counter()
    quality = analyze_page_quality(rendered, cfg)
    view, meta = preprocess_page(rendered.image, quality, cfg)
    timings["preprocessing"] += time.perf_counter() - t0
    return view, quality, meta


def extract_identity_fields(pdf_path: Path, customer_id: str, engine,
                            timings: Dict[str, float], errors: List[Dict[str, Any]],
                            cfg: Config = CFG) -> Dict[str, Any]:
    """Full-page analysis on every page, targeted retries, and a dedicated MRZ task whose
    candidate region may come from ANY page."""
    t_doc = time.perf_counter()
    t0 = time.perf_counter()
    pages = render_pdf_pages(pdf_path, cfg)
    timings["pdf_render"] += time.perf_counter() - t0

    page_records: List[Dict[str, Any]] = []
    all_candidates: List[MRZCandidate] = []
    prepared: Dict[int, Tuple["RenderedPage", Image.Image, Dict[str, Any], Dict[str, Any]]] = {}
    per_page_fields: Dict[int, Dict[str, Any]] = {}
    title_info = None

    for rendered in pages:
        t_page = time.perf_counter()
        prec: Dict[str, Any] = {
            "customer_id": customer_id, "document": IDENTITY_DOC,
            "page_number": rendered.page_number, "width": rendered.width,
            "height": rendered.height, "render_dpi": rendered.render_dpi,
            "render_time": rendered.render_time,
            "native_image_px": list(rendered.native_image_px) if rendered.native_image_px else None,
            "n_model_calls": 0, "inference_time": 0.0, "status": "OK"}
        try:
            view, quality, meta = prepare_view(rendered, timings, cfg)
            prepared[rendered.page_number] = (rendered, view, meta, quality)
            prec.update({"page_quality": quality, "preprocessing": meta,
                         "preprocessing_time": meta.get("preprocessing_time"),
                         "view_dimensions": list(view.size)})
            if cfg.SAVE_DEBUG_IMAGES:
                save_debug(customer_id, IDENTITY_DOC, rendered.page_number, "page",
                           "original_page", _resize_max(rendered.image, 1600), cfg)
                save_debug(customer_id, IDENTITY_DOC, rendered.page_number, "page",
                           "processed_page", view, cfg)
            if quality["is_blank"]:
                prec["status"] = "SKIPPED_BLANK"
                page_records.append(prec)
                continue

            # ---- MRZ candidates on THIS page. Every page is examined; page 2 is not special.
            t0 = time.perf_counter()
            cands = detect_mrz_candidates(view, rendered.page_number, cfg)
            timings["mrz_detection"] += time.perf_counter() - t0
            all_candidates.extend(cands)
            prec["mrz_candidates"] = [asdict(c) for c in cands]
            prec["mrz_candidate_count"] = len(cands)

            # ---- PASS A: transcribe every line verbatim (highest-recall task for a VLM) ----
            genA = run_qwen_ocr(view, LINES_PROMPT, cfg.MAX_NEW_TOKENS_LINES, engine, cfg=cfg,
                                task="lines", customer_id=customer_id, document=IDENTITY_DOC,
                                page=rendered.page_number)
            timings["inference"] += genA.inference_time
            prec["n_model_calls"] += 1
            prec["inference_time"] += genA.inference_time
            lines, lines_meta, okA = parse_lines_response(genA.raw_model_output)
            from_lines = map_lines_to_fields(lines)

            # ---- PASS B: the schema, on the same pixels ----
            schema_fields = [f for f in IDENTITY_FIELDS if f != "mrz"]
            genB = run_qwen_ocr(view, identity_fields_prompt(schema_fields), cfg.MAX_NEW_TOKENS,
                                engine, cfg=cfg, task="identity_fields",
                                customer_id=customer_id, document=IDENTITY_DOC,
                                page=rendered.page_number)
            timings["inference"] += genB.inference_time
            prec["n_model_calls"] += 1
            prec["inference_time"] += genB.inference_time
            parsedB, okB, warnB = parse_json_response(genB.raw_model_output, schema_fields)

            fields: Dict[str, Any] = {}
            for f in schema_fields:
                vb = parsedB.get(f, {}).get("value")
                vl = (from_lines.get(f) or {}).get("value")
                agree = agreement_score(vb, vl)
                # Prefer the schema read; fall back to the label-mapped line when it is absent.
                value = vb if vb is not None else vl
                fields[f] = make_field(
                    f, value, parsedB.get(f, {}).get("confidence", "unreadable"),
                    document=IDENTITY_DOC, page=rendered.page_number, region="full_page",
                    ocr_task="identity_fields" if vb is not None else "lines_label_mapping",
                    preprocessing_variant=";".join(meta.get("applied", [])),
                    agreement=agree, visual_clarity=quality["visual_clarity"],
                    text_height=quality.get("text_height_px_in_view"),
                    raw_output=genB.raw_model_output if vb is not None else genA.raw_model_output,
                    cfg=cfg)
                if agree is not None:
                    fields[f]["cross_pass_agreement"] = agree
                    fields[f]["line_pass_value"] = vl

            # ---- PASS C: targeted retry, ONLY for fields still missing, at higher resolution ----
            missing = [f for f in MANDATORY_FIELDS if f in fields and fields[f]["value"] is None]
            retry = 0
            while missing and retry < cfg.MAX_TARGETED_RETRIES:
                retry += 1
                tiles = page_tiles(view, cfg.PAGE_TILES)
                box = tiles[min(retry - 1, len(tiles) - 1)]
                t0 = time.perf_counter()
                region = extract_region(box, view, rendered, pdf_path, meta,
                                        cfg.REGION_RENDER_DPI, cfg.MAX_IMAGE_DIMENSION, cfg=cfg)
                timings["roi_detection"] += time.perf_counter() - t0
                genC = run_qwen_ocr(region["image"], identity_fields_prompt(missing),
                                    cfg.MAX_NEW_TOKENS_TARGETED, engine, cfg=cfg,
                                    task="identity_fields_retry", customer_id=customer_id,
                                    document=IDENTITY_DOC, page=rendered.page_number,
                                    retry_number=retry)
                timings["inference"] += genC.inference_time
                prec["n_model_calls"] += 1
                prec["inference_time"] += genC.inference_time
                pc, okC, _ = parse_json_response(genC.raw_model_output, missing)
                for f in list(missing):
                    if pc.get(f, {}).get("value") is not None:
                        fields[f] = make_field(
                            f, pc[f]["value"], pc[f]["confidence"], document=IDENTITY_DOC,
                            page=rendered.page_number, region=f"tile_{retry}:{region['bbox']}",
                            ocr_task="identity_fields_retry", preprocessing_variant=region["source"],
                            retry_number=retry, visual_clarity=quality["visual_clarity"],
                            text_height=(quality.get("text_height_px_in_view") or 0) * 1.8,
                            raw_output=genC.raw_model_output, cfg=cfg)
                if cfg.SAVE_DEBUG_ON_FAILURE or cfg.SAVE_DEBUG_IMAGES:
                    save_debug(customer_id, IDENTITY_DOC, rendered.page_number,
                               "identity_name_crop", f"retry_{retry}", region["image"], cfg)
                missing = [f for f in missing if fields[f]["value"] is None]

            per_page_fields[rendered.page_number] = fields
            prec.update({"lines_transcribed": len(lines), "lines_meta": lines_meta,
                         "fields_found": sum(1 for f in fields.values() if f["value"]),
                         "parse_ok_lines": okA, "parse_ok_fields": okB,
                         "retries": retry,
                         "raw_model_output_lines": genA.raw_model_output[:1500],
                         "raw_model_output_fields": genB.raw_model_output[:1500]})
            if genA.error_kind or genB.error_kind:
                prec["status"] = "SYSTEM_ERROR"
                errors.append({"customer_id": customer_id, "document": IDENTITY_DOC,
                               "page": rendered.page_number, "stage": "identity_inference",
                               "error_type": genA.error_kind or genB.error_kind,
                               "error_message": genA.error or genB.error, "traceback": "",
                               "timestamp": utcnow()})
            if title_info is None and rendered.page_number == 1:
                title_info = read_page_title(rendered, view, pdf_path, meta, engine, timings,
                                             customer_id, IDENTITY_DOC, cfg)
                prec["n_model_calls"] += 1
        except Exception as exc:
            prec["status"] = "PAGE_ERROR"
            prec["error"] = f"{type(exc).__name__}: {exc}"
            errors.append({"customer_id": customer_id, "document": IDENTITY_DOC,
                           "page": rendered.page_number, "stage": "identity_page",
                           "error_type": type(exc).__name__, "error_message": str(exc),
                           "traceback": traceback.format_exc(), "timestamp": utcnow()})
            log.error("identity page %d failed for %s: %s", rendered.page_number, customer_id, exc)
        prec["page_total_time"] = round(time.perf_counter() - t_page, 3)
        page_records.append(prec)

    # ---- merge fields across pages: best confidence wins; a tie with different values -> null --
    final: Dict[str, Any] = {}
    conflicts: List[Dict[str, Any]] = []
    for f in IDENTITY_FIELDS:
        if f == "mrz":
            continue
        cands = [(pn, fl[f]) for pn, fl in per_page_fields.items()
                 if fl.get(f, {}).get("value") is not None]
        if not cands:
            final[f] = empty_field(f, IDENTITY_DOC, "not_visible_on_any_page")
            continue
        cands.sort(key=lambda t: -t[1]["confidence"])
        best = cands[0][1]
        rivals = [c for c in cands[1:]
                  if normalize_for_comparison(c[1]["value"]) !=
                     normalize_for_comparison(best["value"])]
        if [c for c in rivals if c[1]["confidence"] >= best["confidence"] - 0.05]:
            conflicts.append({"field": f, "candidates": [{"page": pn, "value": n["value"],
                                                          "confidence": n["confidence"]}
                                                         for pn, n in cands]})
            final[f] = empty_field(f, IDENTITY_DOC, "conflicting_pages")
        else:
            final[f] = best

    mrz_result = extract_mrz(all_candidates, prepared, pdf_path, customer_id, engine,
                             timings, errors, cfg)
    final["mrz"] = mrz_result["field"]
    title = classify_title(title_info["title_text"] if title_info else None,
                           title_info.get("other_headings") if title_info else None,
                           EXPECTED_TITLES[IDENTITY_DOC], cfg) if title_info else \
        {"status": "UNCERTAIN", "reason": "no title read"}

    return {"document": IDENTITY_DOC, "source_pdf": str(pdf_path), "page_count": len(pages),
            "expected_document": EXPECTED_TITLES[IDENTITY_DOC]["label"],
            "detected_document": title.get("observed_title"),
            "document_verification_status": title["status"], "title_check": title,
            "fields": final, "conflicts": conflicts, "mrz": mrz_result["report"],
            "pages": page_records,
            "n_model_calls": sum(p.get("n_model_calls", 0) for p in page_records)
                             + mrz_result["report"].get("n_model_calls", 0),
            "processing_time": round(time.perf_counter() - t_doc, 2)}


print("CELL 25 ready: extract_identity_fields()")

In [ ]:
# =========================================================================
# CELL 26 — DEDICATED MRZ OCR  (spatial localisation first, then read)
# =========================================================================
def extract_mrz(all_candidates: List[MRZCandidate], prepared: Dict[int, Tuple],
                pdf_path: Path, customer_id: str, engine, timings: Dict[str, float],
                errors: List[Dict[str, Any]], cfg: Config = CFG) -> Dict[str, Any]:
    """Select the best MRZ candidate across ALL pages, crop the ENTIRE region, then OCR it.

    The full-page pass never carries the MRZ: a band whose characters are ~15 px tall in a
    downscaled page is unreadable by construction. Retries change the preprocessing VARIANT,
    never merely repeat the same input."""
    report: Dict[str, Any] = {
        "mrz_detected": False, "mrz_candidate_page": None, "mrz_region": None,
        "mrz_detection_method": None, "mrz_detection_score": None,
        "mrz_crop_width": None, "mrz_crop_height": None, "mrz_upscaled_char_height": None,
        "mrz_inference_time": 0.0, "mrz_validation_status": None, "mrz_raw_output": None,
        "mrz_source": None, "attempts": [], "n_model_calls": 0,
        "candidates_examined": len(all_candidates),
        "candidate_pages": sorted({c.page_number for c in all_candidates})}

    best = localize_mrz(all_candidates)
    if best is None or best.page_number not in prepared:
        report["failure_reason"] = "no MRZ-like region found on any page"
        return {"report": report, "field": empty_field("mrz", IDENTITY_DOC, "no_mrz_region")}

    rendered, view, meta, quality = prepared[best.page_number]
    report.update({"mrz_detected": True, "mrz_candidate_page": best.page_number,
                   "mrz_region": list(best.bbox), "mrz_detection_method": best.method,
                   "mrz_detection_score": best.score,
                   "mrz_candidate_metrics": {"n_lines": best.n_lines,
                                             "width_frac": best.width_frac,
                                             "fill": best.fill,
                                             "height_uniformity": best.height_uniformity,
                                             "pitch_regularity": best.pitch_regularity,
                                             "saturation": best.saturation,
                                             "rel_y": best.rel_y}})

    variants = ["standard", "aggressive", "binarised"][:1 + cfg.MAX_TARGETED_RETRIES]
    value, band, raw_out, used = None, "unreadable", "", None
    for attempt, variant in enumerate(variants, start=1):
        t0 = time.perf_counter()
        crop = crop_mrz(best, view, rendered, pdf_path, meta, variant, cfg)
        timings["mrz_preprocessing"] += time.perf_counter() - t0
        if attempt == 1 and (cfg.SAVE_DEBUG_IMAGES or cfg.SAVE_DEBUG_ON_FAILURE):
            save_debug(customer_id, IDENTITY_DOC, best.page_number, "mrz", "mrz_candidate",
                       crop["original"], cfg)
        gen = run_qwen_ocr(crop["image"], MRZ_PROMPT, cfg.MAX_NEW_TOKENS_MRZ, engine, cfg=cfg,
                           task="mrz", customer_id=customer_id, document=IDENTITY_DOC,
                           page=best.page_number, retry_number=attempt - 1)
        timings["inference"] += gen.inference_time
        report["mrz_inference_time"] = round(report["mrz_inference_time"] + gen.inference_time, 3)
        report["n_model_calls"] += 1
        parsed, ok, _ = parse_json_response(gen.raw_model_output, ["mrz"])
        candidate_value = parsed["mrz"]["value"]
        structural = looks_like_mrz(candidate_value)
        validation = validate_mrz(candidate_value)

        report["attempts"].append({
            "attempt": attempt, "variant": variant, "reason": (
                "first pass" if attempt == 1 else
                f"previous attempt produced {'no value' if value is None else 'a non-MRZ string'}; "
                f"switching preprocessing to '{variant}'"),
            "source": crop["source"], "ops": crop["ops"],
            "crop": [crop["crop_width"], crop["crop_height"]],
            "char_height_px": crop["estimated_char_height_px"],
            "sufficient_resolution": crop["sufficient_resolution"],
            "visual_tokens": crop["approx_visual_tokens"],
            "inference_time": gen.inference_time, "parse_ok": ok,
            "structurally_mrz": structural, "validation_status": validation["status"]})
        report.update({"mrz_crop_width": crop["crop_width"], "mrz_crop_height": crop["crop_height"],
                       "mrz_upscaled_char_height": crop["estimated_char_height_px"],
                       "mrz_source": crop["source"], "mrz_raw_output": gen.raw_model_output[:1500],
                       "mrz_validation_status": validation["status"], "mrz_validation": validation})
        if cfg.SAVE_DEBUG_IMAGES or (cfg.SAVE_DEBUG_ON_FAILURE and candidate_value is None):
            save_debug(customer_id, IDENTITY_DOC, best.page_number, "mrz",
                       f"mrz_processed_{variant}", crop["image"], cfg)
        if gen.error_kind:
            errors.append({"customer_id": customer_id, "document": IDENTITY_DOC,
                           "page": best.page_number, "stage": f"mrz_attempt_{attempt}",
                           "error_type": gen.error_kind, "error_message": gen.error,
                           "traceback": "", "timestamp": utcnow()})
            break
        if candidate_value is not None and structural:
            value, band, raw_out, used = candidate_value, parsed["mrz"]["confidence"], \
                gen.raw_model_output, crop
            break
        if candidate_value is not None and not structural:
            # The region was localised but the transcription is not MRZ-shaped: this is the
            # discriminator that rejects Arabic text, printed fields and decoration after the
            # fact. Keep the raw output for audit, do not accept it as an MRZ.
            report["rejected_non_mrz_transcription"] = candidate_value[:200]

    if value is None:
        report["failure_reason"] = ("MRZ region located but not readable after "
                                    f"{len(report['attempts'])} preprocessing variants")
        field = empty_field("mrz", IDENTITY_DOC, "mrz_unreadable")
        field.update({"source_page": best.page_number, "source_region": list(best.bbox),
                      "ocr_task": "mrz", "raw_model_output": (raw_out or "")[:800]})
        return {"report": report, "field": field}

    field = make_field("mrz", value, band, document=IDENTITY_DOC, page=best.page_number,
                       region=f"mrz:{best.bbox}", ocr_task="mrz",
                       preprocessing_variant=used["variant"],
                       retry_number=len(report["attempts"]) - 1,
                       visual_clarity=quality["visual_clarity"],
                       text_height=used["estimated_char_height_px"],
                       raw_output=raw_out, cfg=cfg)
    field["mrz_validation"] = report["mrz_validation"]   # diagnostic only; value is untouched
    return {"report": report, "field": field}


print("CELL 26 ready: extract_mrz()")

In [ ]:
# =========================================================================
# CELL 27 — SECONDARY DOCUMENTS: DOMICILE, CONVENTION, FATCA, SIGNATURE
# =========================================================================
def _extract_names_from_view(rendered: "RenderedPage", view: Image.Image, meta: Dict[str, Any],
                             quality: Dict[str, Any], pdf_path: Path, customer_id: str,
                             document: str, engine, timings: Dict[str, float],
                             handwritten: bool, cfg: Config = CFG
                             ) -> Tuple[Dict[str, Any], int, float]:
    """Full-page name read, then targeted high-resolution crops for whatever is still missing."""
    gen = run_qwen_ocr(view, name_prompt(handwritten), cfg.MAX_NEW_TOKENS_TARGETED, engine,
                       cfg=cfg, task="name", customer_id=customer_id, document=document,
                       page=rendered.page_number)
    timings["inference"] += gen.inference_time
    calls, infer = 1, gen.inference_time
    parsed, ok, _ = parse_json_response(gen.raw_model_output, NAME_FIELDS)

    fields = {f: make_field(f, parsed[f]["value"], parsed[f]["confidence"], document=document,
                            page=rendered.page_number, region="full_page", ocr_task="name",
                            preprocessing_variant=";".join(meta.get("applied", [])),
                            visual_clarity=quality["visual_clarity"],
                            text_height=quality.get("text_height_px_in_view"),
                            text_type=parsed[f].get("text_type"),
                            raw_output=gen.raw_model_output, cfg=cfg)
              for f in NAME_FIELDS}

    missing = [f for f in NAME_FIELDS if fields[f]["value"] is None]
    retry = 0
    while missing and retry < cfg.MAX_TARGETED_RETRIES:
        retry += 1
        tiles = page_tiles(view, cfg.PAGE_TILES)
        box = tiles[min(retry - 1, len(tiles) - 1)]
        t0 = time.perf_counter()
        crop = handwriting_region(box, view, rendered, pdf_path, meta,
                                  "standard" if retry == 1 else "contrast", cfg)
        timings["roi_detection"] += time.perf_counter() - t0
        g2 = run_qwen_ocr(crop["image"], name_prompt(handwritten), cfg.MAX_NEW_TOKENS_TARGETED,
                          engine, cfg=cfg, task="handwriting" if handwritten else "name_retry",
                          customer_id=customer_id, document=document,
                          page=rendered.page_number, retry_number=retry)
        timings["handwriting_inference" if handwritten else "inference"] += g2.inference_time
        calls += 1
        infer += g2.inference_time
        p2, ok2, _ = parse_json_response(g2.raw_model_output, NAME_FIELDS)
        for f in list(missing):
            if p2.get(f, {}).get("value") is not None:
                fields[f] = make_field(f, p2[f]["value"], p2[f]["confidence"], document=document,
                                       page=rendered.page_number,
                                       region=f"tile_{retry}:{crop['bbox']}",
                                       ocr_task="handwriting" if handwritten else "name_retry",
                                       preprocessing_variant=crop["variant"], retry_number=retry,
                                       visual_clarity=quality["visual_clarity"],
                                       text_height=(quality.get("text_height_px_in_view") or 0) * 1.8,
                                       text_type=p2[f].get("text_type"),
                                       raw_output=g2.raw_model_output, cfg=cfg)
        if cfg.SAVE_DEBUG_IMAGES or (cfg.SAVE_DEBUG_ON_FAILURE and missing):
            task = "fatca_handwriting_crop" if document == FATCA_DOC else (
                "signature_handwriting_crop" if document == SIGNATURE_DOC else
                "domicile_name_crop" if document == DOMICILE_DOC else "convention_name_crop")
            save_debug(customer_id, document, rendered.page_number, task,
                       f"retry_{retry}_{crop['variant']}", crop["image"], cfg)
        missing = [f for f in NAME_FIELDS if fields[f]["value"] is None]
    return fields, calls, infer


def extract_secondary_document(customer_id: str, document: str, pdf_path: Path, engine,
                               expected: Dict[str, Any], handwritten: bool,
                               timings: Dict[str, float], errors: List[Dict[str, Any]],
                               cfg: Config = CFG) -> Dict[str, Any]:
    """Verify the document visually, then read the Latin name FROM THIS DOCUMENT ONLY."""
    t_doc = time.perf_counter()
    t0 = time.perf_counter()
    pages = render_pdf_pages(pdf_path, cfg)
    timings["pdf_render"] += time.perf_counter() - t0
    page_records, calls = [], 0
    names = {f: empty_field(f, document, "not_extracted") for f in NAME_FIELDS}
    title_info = None

    for rendered in pages:
        prec = {"customer_id": customer_id, "document": document,
                "page_number": rendered.page_number, "width": rendered.width,
                "height": rendered.height, "render_dpi": rendered.render_dpi,
                "render_time": rendered.render_time, "n_model_calls": 0,
                "inference_time": 0.0, "status": "OK"}
        try:
            view, quality, meta = prepare_view(rendered, timings, cfg)
            prec.update({"page_quality": quality, "preprocessing": meta,
                         "preprocessing_time": meta.get("preprocessing_time")})
            if cfg.SAVE_DEBUG_IMAGES:
                save_debug(customer_id, document, rendered.page_number, "page",
                           "processed_page", view, cfg)
            if quality["is_blank"]:
                prec["status"] = "SKIPPED_BLANK"
                page_records.append(prec)
                continue
            if title_info is None:
                title_info = read_page_title(rendered, view, pdf_path, meta, engine, timings,
                                             customer_id, document, cfg)
                calls += 1
                prec["n_model_calls"] += 1
                prec["title_text"] = title_info["title_text"]
            verdict = classify_title(title_info["title_text"], title_info.get("other_headings"),
                                     expected, cfg)
            # A document that is demonstrably the WRONG document is not a source of identity data.
            if verdict["status"] != "MISMATCH" and any(n["value"] is None for n in names.values()):
                got, c, infer = _extract_names_from_view(rendered, view, meta, quality, pdf_path,
                                                         customer_id, document, engine, timings,
                                                         handwritten, cfg)
                calls += c
                prec["n_model_calls"] += c
                prec["inference_time"] = round(infer, 3)
                for f in NAME_FIELDS:
                    if names[f]["value"] is None and got[f]["value"] is not None:
                        names[f] = got[f]
        except Exception as exc:
            prec["status"] = "PAGE_ERROR"
            prec["error"] = str(exc)
            errors.append({"customer_id": customer_id, "document": document,
                           "page": rendered.page_number, "stage": "secondary_page",
                           "error_type": type(exc).__name__, "error_message": str(exc),
                           "traceback": traceback.format_exc(), "timestamp": utcnow()})
        page_records.append(prec)

    verdict = classify_title(title_info["title_text"] if title_info else None,
                             title_info.get("other_headings") if title_info else None,
                             expected, cfg) if title_info else \
        {"status": "UNCERTAIN", "observed_title": None, "reason": "no page read"}
    return {"document": document, "source_pdf": str(pdf_path), "page_count": len(pages),
            "expected_document": expected.get("label"),
            "detected_document": verdict.get("observed_title"),
            "document_verification_status": verdict["status"], "title_check": verdict,
            "document_title_verified": verdict["status"] == "MATCH",
            "fields": names, "pages": page_records, "n_model_calls": calls,
            "processing_time": round(time.perf_counter() - t_doc, 2)}


def extract_fatca(customer_id: str, pdf_path: Path, engine, timings: Dict[str, float],
                  errors: List[Dict[str, Any]], cfg: Config = CFG) -> Dict[str, Any]:
    """Two components, each verified visually, with the page count of component 1 checked."""
    t_doc = time.perf_counter()
    t0 = time.perf_counter()
    pages = render_pdf_pages(pdf_path, cfg)
    timings["pdf_render"] += time.perf_counter() - t0

    prepared, page_titles, page_records, calls = {}, [], [], 0
    for rendered in pages:
        prec = {"customer_id": customer_id, "document": FATCA_DOC,
                "page_number": rendered.page_number, "width": rendered.width,
                "height": rendered.height, "render_dpi": rendered.render_dpi,
                "render_time": rendered.render_time, "n_model_calls": 0,
                "inference_time": 0.0, "status": "OK"}
        try:
            view, quality, meta = prepare_view(rendered, timings, cfg)
            prepared[rendered.page_number] = (rendered, view, meta, quality)
            prec.update({"page_quality": quality, "preprocessing": meta,
                         "preprocessing_time": meta.get("preprocessing_time")})
            tinfo = {"title_text": None, "other_headings": None}
            if not quality["is_blank"]:
                tinfo = read_page_title(rendered, view, pdf_path, meta, engine, timings,
                                        customer_id, FATCA_DOC, cfg)
                calls += 1
                prec["n_model_calls"] += 1
                prec["inference_time"] += tinfo.get("inference_time", 0.0)
            else:
                prec["status"] = "SKIPPED_BLANK"
            prec["title_text"] = tinfo.get("title_text")
            page_titles.append({"page_number": rendered.page_number, **tinfo})
        except Exception as exc:
            prec["status"] = "PAGE_ERROR"
            errors.append({"customer_id": customer_id, "document": FATCA_DOC,
                           "page": rendered.page_number, "stage": "fatca_page",
                           "error_type": type(exc).__name__, "error_message": str(exc),
                           "traceback": traceback.format_exc(), "timestamp": utcnow()})
        page_records.append(prec)

    components: Dict[str, Any] = {}
    claimed: set = set()
    # component_2 is matched FIRST and claims its pages: its title is the more specific one.
    for key in ("component_2", "component_1"):
        spec = FATCA_COMPONENTS[key]
        hits = []
        for pt in page_titles:
            if pt["page_number"] in claimed:
                continue
            v = classify_title(pt["title_text"], pt.get("other_headings"), spec, cfg)
            if v["status"] == "MATCH":
                hits.append({"page_number": pt["page_number"], **v})
                claimed.add(pt["page_number"])
        detected = bool(hits)
        status = "MATCH" if detected else (
            "UNCERTAIN" if all(not pt["title_text"] for pt in page_titles) else "MISSING")
        components[key] = {
            "expected_title": spec["label"], "detected": detected, "title_verified": detected,
            "status": status, "pages_found": [h["page_number"] for h in hits],
            "page_count": len(hits), "expected_pages": spec["expected_pages"],
            "page_count_verified": (len(hits) == spec["expected_pages"]) if detected else False,
            "observed_titles": [pt["title_text"] for pt in page_titles]}
        if detected and len(hits) != spec["expected_pages"]:
            components[key]["flag"] = (f"expected {spec['expected_pages']} page(s), "
                                       f"found {len(hits)}")

    names = {f: empty_field(f, FATCA_DOC, "not_extracted") for f in NAME_FIELDS}
    targets = components["component_2"]["pages_found"] or (
        [p["page_number"] for p in page_titles] if components["component_2"]["status"] ==
        "UNCERTAIN" else [])
    for pn in targets:
        if pn not in prepared:
            continue
        rendered, view, meta, quality = prepared[pn]
        try:
            got, c, infer = _extract_names_from_view(rendered, view, meta, quality, pdf_path,
                                                     customer_id, FATCA_DOC, engine, timings,
                                                     True, cfg)
            calls += c
            for f in NAME_FIELDS:
                if names[f]["value"] is None and got[f]["value"] is not None:
                    names[f] = got[f]
            if all(n["value"] is not None for n in names.values()):
                break
        except Exception as exc:
            errors.append({"customer_id": customer_id, "document": FATCA_DOC, "page": pn,
                           "stage": "fatca_handwriting", "error_type": type(exc).__name__,
                           "error_message": str(exc), "traceback": traceback.format_exc(),
                           "timestamp": utcnow()})

    overall = ("MATCH" if all(c["status"] == "MATCH" for c in components.values()) else
               "UNCERTAIN" if any(c["status"] == "UNCERTAIN" for c in components.values())
               else "MISMATCH")
    return {"document": FATCA_DOC, "source_pdf": str(pdf_path), "page_count": len(pages),
            "expected_document": "FATCA identification forms (two components)",
            "detected_document": "; ".join(t for t in
                                           [pt["title_text"] for pt in page_titles] if t) or None,
            "document_verification_status": overall,
            "component_1_detected": components["component_1"]["detected"],
            "component_1_header_verified": components["component_1"]["title_verified"],
            "component_1_page_count": components["component_1"]["page_count"],
            "component_2_detected": components["component_2"]["detected"],
            "component_2_title_verified": components["component_2"]["title_verified"],
            "components": components, "fields": names, "pages": page_records,
            "n_model_calls": calls, "processing_time": round(time.perf_counter() - t_doc, 2)}


print("CELL 27 ready: extract_secondary_document(), extract_fatca()")

In [ ]:
# =========================================================================
# CELL 28 — CROSS-DOCUMENT COMPARISON  (after independent extraction, never before)
# =========================================================================
# Comparison NEVER modifies OCR. It only produces a verdict.
#   MATCH          both sides read reliably and identical after comparison-only normalisation
#   MISMATCH       both sides read reliably and different
#   UNCERTAIN      at least one side is low confidence, or contains ? characters
#   NOT_AVAILABLE  at least one side is null
# Unreadable is NEVER a mismatch.
MATCH, MISMATCH, UNCERTAIN, NOT_AVAILABLE = "MATCH", "MISMATCH", "UNCERTAIN", "NOT_AVAILABLE"
RELIABLE_BANDS = ("high", "medium")


def compare_values(doc_value: Optional[str], ref_value: Optional[str],
                   doc_band: str = "unreadable", method: str = "normalized_exact",
                   cfg: Config = CFG) -> Dict[str, Any]:
    a, b = normalize_for_comparison(doc_value), normalize_for_comparison(ref_value)
    out = {"document_value": doc_value, "reference_value": ref_value,
           "normalized_document_value": a, "normalized_reference_value": b,
           "comparison_method": method, "similarity_score": None,
           "threshold": cfg.NAME_FUZZY_THRESHOLD, "document_confidence_band": doc_band}
    if a is None or b is None:
        return {**out, "status": NOT_AVAILABLE,
                "reason": "value not available on one side -- unreadable is not a mismatch"}
    if doc_value and "?" in str(doc_value):
        return {**out, "status": UNCERTAIN, "reason": "document value has unreadable characters"}
    if a == b:
        return {**out, "status": MATCH if doc_band in RELIABLE_BANDS else UNCERTAIN,
                "similarity_score": 1.0,
                "reason": "identical after normalisation" if doc_band in RELIABLE_BANDS
                else "values agree but the document read is not reliable"}
    score = round(difflib.SequenceMatcher(None, a, b).ratio(), 3)
    out["similarity_score"] = score
    if doc_band not in RELIABLE_BANDS:
        return {**out, "status": UNCERTAIN, "comparison_method": "difflib_ratio",
                "reason": f"values differ (score {score}) and the document read is not reliable"}
    if score >= cfg.NAME_FUZZY_THRESHOLD:
        # Deliberately NOT reported as MATCH: a near-identical string is a reason for a human to
        # look, never a confirmation. The score and threshold are always exposed.
        return {**out, "status": UNCERTAIN, "comparison_method": "difflib_ratio",
                "reason": f"near match {score} >= {cfg.NAME_FUZZY_THRESHOLD} but not identical"}
    return {**out, "status": MISMATCH, "comparison_method": "difflib_ratio",
            "reason": f"values differ, score {score} < {cfg.NAME_FUZZY_THRESHOLD}"}


def compare_documents(documents: Dict[str, Optional[Dict[str, Any]]],
                      cfg: Config = CFG) -> Dict[str, Any]:
    """Identity is the reference, but agreement with it never overrides uncertainty elsewhere."""
    identity = documents.get(IDENTITY_DOC) or {}
    ref = identity.get("fields", {})
    rows, details = [], []
    for doc in DOCUMENTS_REQUIRED:
        res = documents.get(doc)
        if doc == IDENTITY_DOC:
            rows.append({"document": doc, "surname_latin": "reference",
                         "given_names_latin": "reference", "overall": "reference",
                         "document_verification_status":
                             (res or {}).get("document_verification_status", "NOT_AVAILABLE")})
            continue
        if res is None:
            rows.append({"document": doc, "surname_latin": NOT_AVAILABLE,
                         "given_names_latin": NOT_AVAILABLE, "overall": NOT_AVAILABLE,
                         "document_verification_status": "MISSING"})
            continue
        statuses = {}
        for f in NAME_FIELDS:
            node = res.get("fields", {}).get(f, {})
            refnode = ref.get(f, {})
            cmp_ = compare_values(node.get("value"), refnode.get("value"),
                                  node.get("confidence_band", "unreadable"), cfg=cfg)
            cmp_.update({"source_document": doc, "reference_source": IDENTITY_DOC, "field": f})
            statuses[f] = cmp_["status"]
            details.append(cmp_)
        overall = (MISMATCH if MISMATCH in statuses.values() else
                   UNCERTAIN if UNCERTAIN in statuses.values() else
                   NOT_AVAILABLE if all(s == NOT_AVAILABLE for s in statuses.values()) else MATCH)
        rows.append({"document": doc, **statuses, "overall": overall,
                     "document_verification_status": res.get("document_verification_status")})
    return {"matrix": rows, "comparisons": details}


print("CELL 28 ready: compare_documents()")

In [ ]:
# =========================================================================
# CELL 29 — DATABASE VERIFICATION  (four columns, comparison only)
# =========================================================================
def _name_tokens(value: Optional[str]) -> Optional[List[str]]:
    n = normalize_for_comparison(value)
    return sorted(n.split()) if n else None


def compare_name_any_order(doc_surname: Optional[str], doc_given: Optional[str],
                           db_name: Optional[str], doc_band: str,
                           cfg: Config = CFG) -> Dict[str, Any]:
    """`Nom abrégé tiers` may be LAST FIRST or FIRST LAST. Compare as token multisets so both
    orders match, and report which order was observed. Normalisation is for comparison only."""
    parts = [p for p in (doc_surname, doc_given) if p]
    doc_full = " ".join(parts) if parts else None
    out = {"document_value": doc_full, "reference_value": db_name,
           "normalized_document_value": normalize_for_comparison(doc_full),
           "normalized_reference_value": normalize_for_comparison(db_name),
           "comparison_method": "token_multiset_any_order",
           "similarity_score": None, "threshold": cfg.NAME_FUZZY_THRESHOLD,
           "document_confidence_band": doc_band, "observed_order": None}
    dt, rt = _name_tokens(doc_full), _name_tokens(db_name)
    if dt is None or rt is None:
        return {**out, "status": NOT_AVAILABLE,
                "reason": "name not available on one side -- unreadable is not a mismatch"}
    if doc_full and "?" in doc_full:
        return {**out, "status": UNCERTAIN, "reason": "document name has unreadable characters"}
    if dt == rt:
        order = None
        if doc_surname and db_name:
            nb = normalize_for_comparison(db_name) or ""
            ns = normalize_for_comparison(doc_surname) or ""
            order = "LAST_FIRST" if nb.startswith(ns) else "FIRST_LAST"
        return {**out, "status": MATCH if doc_band in RELIABLE_BANDS else UNCERTAIN,
                "similarity_score": 1.0, "observed_order": order,
                "reason": "same name tokens in either order" if doc_band in RELIABLE_BANDS
                else "tokens agree but the document read is not reliable"}
    score = round(difflib.SequenceMatcher(None, " ".join(dt), " ".join(rt)).ratio(), 3)
    out["similarity_score"] = score
    if doc_band not in RELIABLE_BANDS:
        return {**out, "status": UNCERTAIN, "reason": "document read is not reliable"}
    if score >= cfg.NAME_FUZZY_THRESHOLD:
        return {**out, "status": UNCERTAIN,
                "reason": f"near match {score} >= {cfg.NAME_FUZZY_THRESHOLD} but tokens differ"}
    return {**out, "status": MISMATCH, "reason": f"name tokens differ, score {score}"}


def compare_dates(doc_value: Optional[str], db_value: Optional[str], doc_band: str,
                  cfg: Config = CFG) -> Dict[str, Any]:
    """Format differences are tolerated by parsing BOTH sides. The extracted representation is
    preserved exactly; only the comparison uses the parsed form."""
    d1, d2 = parse_date(doc_value), parse_date(db_value)
    out = {"document_value": doc_value, "reference_value": db_value,
           "normalized_document_value": d1.isoformat() if d1 else None,
           "normalized_reference_value": d2.isoformat() if d2 else None,
           "comparison_method": "parsed_date_equality", "similarity_score": None,
           "document_confidence_band": doc_band}
    if not doc_value or not db_value:
        return {**out, "status": NOT_AVAILABLE, "reason": "date not available on one side"}
    if "?" in str(doc_value):
        return {**out, "status": UNCERTAIN, "reason": "date has unreadable characters"}
    if d1 is None or d2 is None:
        return {**out, "status": UNCERTAIN,
                "reason": "a date could not be parsed for comparison; raw values preserved"}
    if d1 == d2:
        return {**out, "status": MATCH if doc_band in RELIABLE_BANDS else UNCERTAIN,
                "similarity_score": 1.0, "reason": "same calendar date"}
    return {**out, "status": MISMATCH if doc_band in RELIABLE_BANDS else UNCERTAIN,
            "reason": "different calendar dates"}


def compare_database(customer_id: str, documents: Dict[str, Optional[Dict[str, Any]]],
                     db_row: Optional[Dict[str, str]], cfg: Config = CFG) -> Dict[str, Any]:
    """ONLY Id tiers, Nom abrégé tiers, Date de naissance, Date d'expiration du Document."""
    identity = (documents.get(IDENTITY_DOC) or {}).get("fields", {})

    def node(f):
        return identity.get(f, {}) or {}

    if db_row is None:
        na = {"document_value": None, "reference_value": None, "status": NOT_AVAILABLE,
              "comparison_method": "none", "reason": "no database record for this customer"}
        return {"customer_id": customer_id, "db_id": None,
                "database_comparison": {"customer_id": na, "name": na,
                                        "date_of_birth": na, "expiry_date": na},
                "overall": NOT_AVAILABLE}

    db_id = db_row.get("customer_id")
    id_cmp = {"document_value": customer_id, "reference_value": db_id,
              "comparison_method": "exact_string", "similarity_score": None,
              "status": MATCH if str(customer_id).strip() == str(db_id).strip() else MISMATCH,
              "reason": "folder name compared with Id tiers"}

    name_cmp = compare_name_any_order(node("surname_latin").get("value"),
                                      node("given_names_latin").get("value"),
                                      db_row.get("name"),
                                      node("surname_latin").get("confidence_band", "unreadable"),
                                      cfg)
    dob_cmp = compare_dates(node("date_of_birth").get("value"), db_row.get("date_of_birth"),
                            node("date_of_birth").get("confidence_band", "unreadable"), cfg)
    exp_cmp = compare_dates(node("expiry_date").get("value"), db_row.get("expiry_date"),
                            node("expiry_date").get("confidence_band", "unreadable"), cfg)

    statuses = [id_cmp["status"], name_cmp["status"], dob_cmp["status"], exp_cmp["status"]]
    overall = (MISMATCH if MISMATCH in statuses else
               UNCERTAIN if UNCERTAIN in statuses else
               MATCH if MATCH in statuses else NOT_AVAILABLE)
    return {"customer_id": customer_id, "db_id": db_id,
            "database_comparison": {"customer_id": id_cmp, "name": name_cmp,
                                    "date_of_birth": dob_cmp, "expiry_date": exp_cmp},
            "overall": overall}


print("CELL 29 ready: compare_database() [four authorised columns only]")

In [ ]:
# =========================================================================
# CELL 30 — CUSTOMER AGGREGATION
# =========================================================================
SECONDARY_SPECS = {
    DOMICILE_DOC: {"expected": EXPECTED_TITLES[DOMICILE_DOC], "handwritten": True},
    CONVENTION_DOC: {"expected": EXPECTED_TITLES[CONVENTION_DOC], "handwritten": True},
    SIGNATURE_DOC: {"expected": EXPECTED_TITLES[SIGNATURE_DOC], "handwritten": True},
}


def overall_kyc_status(documents, matrix, db_check, cust, cfg: Config = CFG) -> Dict[str, Any]:
    """PASS / MISMATCH / UNCERTAIN / INCOMPLETE. Unreadable is never a mismatch."""
    reasons: List[str] = []
    missing_docs = [d for d in DOCUMENTS_REQUIRED if not cust.documents.get(d)]
    identity = documents.get(IDENTITY_DOC)
    if identity is None:
        return {"overall_kyc_status": "INCOMPLETE", "processing_status": "no_identity_document",
                "reasons": ["identity document missing or unprocessable"],
                "needs_manual_review": True}

    mismatch_docs = [r["document"] for r in matrix["matrix"] if r.get("overall") == MISMATCH]
    wrong_type = [r["document"] for r in matrix["matrix"]
                  if r.get("document_verification_status") == "MISMATCH"]
    db_status = db_check.get("overall")
    unreadable_mandatory = [f for f in MANDATORY_FIELDS
                            if (identity["fields"].get(f) or {}).get("value") is None]
    mrz = identity.get("mrz", {})

    if mismatch_docs:
        reasons.append("name_mismatch_vs_identity:" + ",".join(mismatch_docs))
    if wrong_type:
        reasons.append("wrong_document_type:" + ",".join(wrong_type))
    if db_status == MISMATCH:
        bad = [k for k, v in db_check["database_comparison"].items() if v["status"] == MISMATCH]
        reasons.append("database_mismatch:" + ",".join(bad))
    if unreadable_mandatory:
        reasons.append("unreadable_mandatory_fields:" + ",".join(unreadable_mandatory))
    if mrz.get("mrz_detected") and not (identity["fields"].get("mrz") or {}).get("value"):
        reasons.append("mrz_region_found_but_unreadable")
    if (identity["fields"].get("mrz") or {}).get("mrz_validation", {}).get("status") == \
            "checksum_mismatch":
        reasons.append("mrz_checksum_mismatch")
    if missing_docs:
        reasons.append("missing_documents:" + ",".join(missing_docs))

    if mismatch_docs or wrong_type or db_status == MISMATCH:
        status = "MISMATCH"
    elif missing_docs:
        status = "INCOMPLETE"
    elif unreadable_mandatory or db_status in (UNCERTAIN, NOT_AVAILABLE) or \
            any(r.get("overall") == UNCERTAIN for r in matrix["matrix"]):
        status = "UNCERTAIN"
    else:
        status = "PASS"

    clarity = [p.get("page_quality", {}).get("visual_clarity")
               for d in documents.values() if d for p in d.get("pages", [])
               if p.get("page_quality")]
    return {"overall_kyc_status": status, "processing_status": "completed",
            "reasons": reasons, "needs_manual_review": status != "PASS",
            "overall_quality": round(float(np.mean(clarity)), 3) if clarity else None}


def aggregate_customer_result(cust: "CustomerDocs", engine, db: Optional[pd.DataFrame],
                              cfg: Config = CFG) -> Dict[str, Any]:
    """STAGE A (independent extraction) then STAGE B (comparison). Never the reverse."""
    t0 = time.perf_counter()
    timings: Dict[str, float] = defaultdict(float)
    errors: List[Dict[str, Any]] = []
    documents: Dict[str, Optional[Dict[str, Any]]] = {d: None for d in DOCUMENTS_REQUIRED}

    try:
        documents[IDENTITY_DOC] = extract_identity_fields(cust.path(IDENTITY_DOC),
                                                          cust.customer_id, engine,
                                                          timings, errors, cfg)
    except Exception as exc:
        errors.append({"customer_id": cust.customer_id, "document": IDENTITY_DOC, "page": None,
                       "stage": "identity_document", "error_type": type(exc).__name__,
                       "error_message": str(exc), "traceback": traceback.format_exc(),
                       "timestamp": utcnow()})
        log.error("identity failed for %s: %s", cust.customer_id, exc)

    if cfg.PROCESS_SECONDARY_DOCUMENTS:
        for doc, spec in SECONDARY_SPECS.items():
            p = cust.path(doc)
            if not p:
                continue
            try:
                documents[doc] = extract_secondary_document(cust.customer_id, doc, p, engine,
                                                            spec["expected"], spec["handwritten"],
                                                            timings, errors, cfg)
            except Exception as exc:
                errors.append({"customer_id": cust.customer_id, "document": doc, "page": None,
                               "stage": "secondary_document", "error_type": type(exc).__name__,
                               "error_message": str(exc), "traceback": traceback.format_exc(),
                               "timestamp": utcnow()})
        if cust.path(FATCA_DOC):
            try:
                documents[FATCA_DOC] = extract_fatca(cust.customer_id, cust.path(FATCA_DOC),
                                                     engine, timings, errors, cfg)
            except Exception as exc:
                errors.append({"customer_id": cust.customer_id, "document": FATCA_DOC,
                               "page": None, "stage": "fatca_document",
                               "error_type": type(exc).__name__, "error_message": str(exc),
                               "traceback": traceback.format_exc(), "timestamp": utcnow()})

    # ---------------- STAGE B ----------------
    t0b = time.perf_counter()
    matrix = compare_documents(documents, cfg)
    timings["cross_document"] += time.perf_counter() - t0b
    t0b = time.perf_counter()
    db_check = compare_database(cust.customer_id, documents, database_record(db, cust.customer_id),
                                cfg)
    timings["database_matching"] += time.perf_counter() - t0b
    verdict = overall_kyc_status(documents, matrix, db_check, cust, cfg)

    by_doc = defaultdict(dict)
    for c in matrix["comparisons"]:
        by_doc[c["source_document"]][c["field"]] = c["status"]
    for doc, res in documents.items():
        if res:
            for f, node in res.get("fields", {}).items():
                if f in NAME_FIELDS:
                    node["match_status"] = by_doc.get(doc, {}).get(f)

    identity = documents.get(IDENTITY_DOC) or {}
    return {"customer_id": cust.customer_id, "run_id": RUN_ID, "extracted_at_utc": utcnow(),
            "engine": {"model": getattr(engine, "name", "?"),
                       "attention": getattr(engine, "attn", "?"),
                       "quantization": getattr(engine, "quantization", {}),
                       "max_image_dimension": cfg.MAX_IMAGE_DIMENSION,
                       "pdf_render_dpi": cfg.PDF_RENDER_DPI,
                       "region_render_dpi": cfg.REGION_RENDER_DPI},
            "document_presence": {PRESENCE_COLUMNS[d]: bool(cust.documents.get(d))
                                  for d in DOCUMENTS_REQUIRED},
            "documents": documents,
            "identity_fields": identity.get("fields", {}),
            "mrz": identity.get("mrz", {"mrz_detected": False}),
            "cross_document_verification": matrix["matrix"],
            "cross_document_comparisons": matrix["comparisons"],
            "database_verification": db_check,
            **verdict,
            "page_count_total": sum((d or {}).get("page_count", 0) for d in documents.values()),
            "n_model_calls": sum((d or {}).get("n_model_calls", 0) for d in documents.values()),
            "timings_seconds": {k: round(v, 3) for k, v in timings.items()},
            "total_customer_time": round(time.perf_counter() - t0, 2),
            "errors": errors}


print("CELL 30 ready: aggregate_customer_result()")

In [ ]:
# =========================================================================
# CELL 31 — BATCH EXECUTION
# =========================================================================
# Batching note: micro-batching several images in one generate() call was evaluated and is NOT
# used. The inputs here differ in size (full page vs MRZ strip vs handwriting crop), in prompt
# and in generation length, so a batch runs at the pace of its slowest member and pads the rest;
# and mixing customers in one call is exactly the isolation risk this pipeline must not take.
# Sequential inference with a persistent model keeps determinism, isolation and auditability.
# On an H100 the win from batching heterogeneous OCR crops is small; the reliability cost is not.
def run_batch(targets: List["CustomerDocs"], engine=None, db: Optional[pd.DataFrame] = None,
              cfg: Config = CFG) -> pd.DataFrame:
    engine = engine or ENGINE
    db = db if db is not None else DB
    rows, consecutive_errors = [], 0
    t_start = time.perf_counter()

    for i, cust in enumerate(targets, 1):
        dest = DIRS["results"] / f"{cust.customer_id}.json"
        if cfg.RESUME and dest.exists():
            log.info("[%d/%d] %s skipped (already done)", i, len(targets), cust.customer_id)
            continue
        try:
            rec = aggregate_customer_result(cust, engine, db, cfg)
        except Exception as exc:                 # one document must never stop the batch
            log.error("[%d/%d] %s FAILED: %s", i, len(targets), cust.customer_id, exc)
            rec = {"customer_id": cust.customer_id, "overall_kyc_status": "INCOMPLETE",
                   "processing_status": "error", "needs_manual_review": True,
                   "reasons": [str(exc)], "documents": {}, "identity_fields": {},
                   "cross_document_verification": [], "database_verification": {},
                   "mrz": {}, "n_model_calls": 0, "page_count_total": 0,
                   "total_customer_time": 0.0, "timings_seconds": {},
                   "errors": [{"customer_id": cust.customer_id, "document": None, "page": None,
                               "stage": "aggregate_customer_result",
                               "error_type": type(exc).__name__, "error_message": str(exc),
                               "traceback": traceback.format_exc(), "timestamp": utcnow()}]}
        dest.write_text(json.dumps(rec, ensure_ascii=False, indent=2, default=str),
                        encoding="utf-8")

        mrz = rec.get("mrz") or {}
        mrz_val = (rec.get("identity_fields", {}).get("mrz") or {}).get("value")
        log.info("[%d/%d] %s %s | %dp %.1fs calls=%d | mrz: page=%s read=%s | db=%s",
                 i, len(targets), cust.customer_id, rec.get("overall_kyc_status"),
                 rec.get("page_count_total", 0), rec.get("total_customer_time", 0),
                 rec.get("n_model_calls", 0), mrz.get("mrz_candidate_page"),
                 bool(mrz_val), (rec.get("database_verification") or {}).get("overall"))
        rows.append({"customer_id": cust.customer_id,
                     "overall_kyc_status": rec.get("overall_kyc_status"),
                     "pages": rec.get("page_count_total", 0),
                     "time_s": rec.get("total_customer_time"),
                     "model_calls": rec.get("n_model_calls"),
                     "mrz_page": mrz.get("mrz_candidate_page"),
                     "mrz_read": bool(mrz_val),
                     "mrz_validation": mrz.get("mrz_validation_status"),
                     "database": (rec.get("database_verification") or {}).get("overall"),
                     "needs_manual_review": rec.get("needs_manual_review")})

        if rec.get("processing_status") == "error":
            consecutive_errors += 1
            if consecutive_errors >= cfg.MAX_CONSECUTIVE_ERRORS:
                log.error("ABORTING BATCH after %d consecutive failures. Run gpu_report(); "
                          "free_model() or restart the kernel; raise CFG.RESERVE_VRAM_GIB or "
                          "lower CFG.MAX_VISUAL_TOKENS.", consecutive_errors)
                break
        else:
            consecutive_errors = 0

    record_memory("after_batch")
    log.info("batch finished in %.1f min", (time.perf_counter() - t_start) / 60)
    return pd.DataFrame(rows)


print("CELL 31 ready: run_batch()")

In [ ]:
# =========================================================================
# CELL 32 — PERFORMANCE LOGGING AND OUTPUT GENERATION
# =========================================================================
def load_results(results_dir: Path = None) -> List[Dict[str, Any]]:
    out = []
    for p in sorted(Path(results_dir or DIRS["results"]).glob("*.json")):
        try:
            out.append(json.loads(p.read_text(encoding="utf-8")))
        except Exception as exc:
            log.warning("unreadable result %s: %s", p.name, exc)
    return out


def build_processing_log(records: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for r in records:
        for doc, res in (r.get("documents") or {}).items():
            if not res:
                continue
            for p in res.get("pages", []):
                rows.append({
                    "customer_id": r["customer_id"], "document": doc,
                    "page_number": p.get("page_number"),
                    "width": p.get("width"), "height": p.get("height"),
                    "render_dpi": p.get("render_dpi"), "render_time": p.get("render_time"),
                    "preprocessing_time": p.get("preprocessing_time"),
                    "inference_time": p.get("inference_time"),
                    "n_model_calls": p.get("n_model_calls"),
                    "mrz_candidates": p.get("mrz_candidate_count"),
                    "quality": p.get("page_quality", {}).get("quality"),
                    "text_height_in_view": p.get("page_quality", {}).get("text_height_px_in_view"),
                    "rotation": (p.get("preprocessing") or {}).get("rotation"),
                    "lines_transcribed": p.get("lines_transcribed"),
                    "fields_found": p.get("fields_found"),
                    "retries": p.get("retries"),
                    "status": p.get("status"),
                    "page_total_time": p.get("page_total_time")})
    return pd.DataFrame(rows)


def flatten_customer(r: Dict[str, Any]) -> Dict[str, Any]:
    row: Dict[str, Any] = OrderedDict(customer_id=r["customer_id"],
                                      overall_kyc_status=r.get("overall_kyc_status"),
                                      processing_status=r.get("processing_status"),
                                      needs_manual_review=r.get("needs_manual_review"),
                                      overall_quality=r.get("overall_quality"),
                                      reasons=";".join(r.get("reasons") or []),
                                      page_count_total=r.get("page_count_total"),
                                      total_customer_time=r.get("total_customer_time"),
                                      n_model_calls=r.get("n_model_calls"))
    row.update(r.get("document_presence") or {})
    for f in IDENTITY_FIELDS:
        n = (r.get("identity_fields") or {}).get(f) or {}
        row[f] = n.get("value")
        row[f + "_confidence"] = n.get("confidence")
        row[f + "_band"] = n.get("confidence_band")
        row[f + "_page"] = n.get("source_page")
        row[f + "_task"] = n.get("ocr_task")
    mrz = r.get("mrz") or {}
    row.update({"mrz_detected": mrz.get("mrz_detected"),
                "mrz_candidate_page": mrz.get("mrz_candidate_page"),
                "mrz_detection_method": mrz.get("mrz_detection_method"),
                "mrz_detection_score": mrz.get("mrz_detection_score"),
                "mrz_crop_width": mrz.get("mrz_crop_width"),
                "mrz_crop_height": mrz.get("mrz_crop_height"),
                "mrz_char_height_px": mrz.get("mrz_upscaled_char_height"),
                "mrz_inference_time": mrz.get("mrz_inference_time"),
                "mrz_validation_status": mrz.get("mrz_validation_status"),
                "mrz_source": mrz.get("mrz_source")})
    for x in r.get("cross_document_verification") or []:
        key = re.sub(r"[^a-z]+", "_", x["document"].lower()).strip("_")
        row[f"identity_vs_{key}_name"] = x.get("overall")
    dbc = (r.get("database_verification") or {}).get("database_comparison") or {}
    for k in ("customer_id", "name", "date_of_birth", "expiry_date"):
        row[f"db_{k}_match"] = (dbc.get(k) or {}).get("status")
    row["errors"] = len(r.get("errors") or [])
    return row


def write_outputs(records: List[Dict[str, Any]] = None, cfg: Config = CFG) -> Dict[str, Path]:
    records = records if records is not None else load_results()
    rep, paths = DIRS["reports"], {}

    with (rep / "identity_extraction_results.jsonl").open("w", encoding="utf-8") as fh:
        for r in records:
            fh.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
    paths["jsonl"] = rep / "identity_extraction_results.jsonl"

    flat = pd.DataFrame([flatten_customer(r) for r in records]) if records else pd.DataFrame()
    flat.to_csv(rep / "identity_extraction_results.csv", index=False, encoding="utf-8-sig")
    paths["identity_csv"] = rep / "identity_extraction_results.csv"

    xdoc = [{"customer_id": r["customer_id"], **m} for r in records
            for m in (r.get("cross_document_verification") or [])]
    pd.DataFrame(xdoc).to_csv(rep / "cross_document_verification.csv", index=False,
                              encoding="utf-8-sig")
    paths["cross_document"] = rep / "cross_document_verification.csv"

    dbrows = []
    for r in records:
        d = r.get("database_verification") or {}
        c = d.get("database_comparison") or {}
        dbrows.append({
            "customer_id": r["customer_id"], "db_id": d.get("db_id"),
            "id_match_status": (c.get("customer_id") or {}).get("status"),
            "ocr_full_name": (c.get("name") or {}).get("document_value"),
            "db_full_name": (c.get("name") or {}).get("reference_value"),
            "name_match_status": (c.get("name") or {}).get("status"),
            "name_similarity": (c.get("name") or {}).get("similarity_score"),
            "name_observed_order": (c.get("name") or {}).get("observed_order"),
            "ocr_date_of_birth": (c.get("date_of_birth") or {}).get("document_value"),
            "db_date_of_birth": (c.get("date_of_birth") or {}).get("reference_value"),
            "dob_match_status": (c.get("date_of_birth") or {}).get("status"),
            "ocr_expiry_date": (c.get("expiry_date") or {}).get("document_value"),
            "db_expiry_date": (c.get("expiry_date") or {}).get("reference_value"),
            "expiry_match_status": (c.get("expiry_date") or {}).get("status"),
            "overall": d.get("overall")})
    pd.DataFrame(dbrows).to_csv(rep / "database_verification.csv", index=False,
                                encoding="utf-8-sig")
    paths["database"] = rep / "database_verification.csv"

    final_cols = ["customer_id", "overall_kyc_status", "needs_manual_review", "overall_quality",
                  "reasons", "surname_latin", "given_names_latin", "date_of_birth",
                  "expiry_date", "document_number", "mrz", "mrz_candidate_page",
                  "db_name_match", "db_date_of_birth_match", "db_expiry_date_match",
                  "db_customer_id_match", "errors"]
    if len(flat):
        flat[[c for c in final_cols if c in flat.columns]].to_csv(
            rep / "final_kyc_results.csv", index=False, encoding="utf-8-sig")
    paths["final"] = rep / "final_kyc_results.csv"

    build_processing_log(records).to_csv(rep / "processing_log.csv", index=False,
                                         encoding="utf-8-sig")
    paths["processing_log"] = rep / "processing_log.csv"
    pd.DataFrame(INFERENCE_LOG).to_csv(rep / "inference_log.csv", index=False,
                                       encoding="utf-8-sig")
    paths["inference_log"] = rep / "inference_log.csv"

    errs = [e for r in records for e in (r.get("errors") or [])]
    pd.DataFrame(errs, columns=["customer_id", "document", "page", "stage", "error_type",
                                "error_message", "traceback", "timestamp"]).to_csv(
        rep / "errors.csv", index=False, encoding="utf-8-sig")
    paths["errors"] = rep / "errors.csv"

    for k, v in paths.items():
        print(f"  {k:<18}: {v}")
    if len(flat):
        print("\nKYC status:", dict(flat["overall_kyc_status"].value_counts()))
        print("MRZ read  :", int(flat["mrz"].notna().sum()), "/", len(flat))
    return paths


print("CELL 32 ready: write_outputs()")

In [ ]:
# =========================================================================
# CELL 33 — BENCHMARK  (warm-up excluded from steady state)
# =========================================================================
def benchmark_pipeline(records: List[Dict[str, Any]] = None, cfg: Config = CFG) -> Dict[str, Any]:
    records = records if records is not None else load_results()
    if not records:
        print("no results yet")
        return {}
    inf = pd.DataFrame(INFERENCE_LOG)
    steady = inf.iloc[1:] if len(inf) > 1 else inf     # drop the first call: warm-up

    def st(series):
        a = pd.to_numeric(pd.Series(series), errors="coerce").dropna()
        return {"mean": None, "median": None, "p95": None} if a.empty else {
            "mean": round(float(a.mean()), 3), "median": round(float(a.median()), 3),
            "p95": round(float(a.quantile(0.95)), 3)}

    tim = [r.get("timings_seconds", {}) for r in records]
    stage_keys = ["pdf_render", "preprocessing", "roi_detection", "mrz_detection",
                  "mrz_preprocessing", "inference", "handwriting_inference",
                  "cross_document", "database_matching"]
    totals = {k: sum(t.get(k, 0) or 0 for t in tim) for k in stage_keys}
    grand = sum(totals.values()) or 1.0
    total_time = sum(r.get("total_customer_time", 0) or 0 for r in records) or 1e-9
    pages = sum(r.get("page_count_total", 0) or 0 for r in records)

    by_task = {}
    if not steady.empty and "task" in steady.columns:
        for task, grp in steady.groupby("task"):
            by_task[task] = {"n": int(len(grp)),
                             "inference_time": st(grp["inference_time"]),
                             "generation_time": st(grp["generation_time"]),
                             "output_tokens": st(grp["output_tokens"])}

    out = {
        "n_customers": len(records), "n_pages": pages,
        "warm_up": WARMUP_STATS,
        "documents_per_second": round(len(records) / total_time, 4),
        "pages_per_second": round(pages / total_time, 4),
        "avg_document_time": round(total_time / max(1, len(records)), 2),
        "per_stage_totals_s": {k: round(v, 2) for k, v in totals.items()},
        "share_of_measured_time_pct": {k: round(100 * v / grand, 1)
                                       for k, v in sorted(totals.items(), key=lambda x: -x[1])},
        "steady_state_inference": st(steady["inference_time"]) if not steady.empty else {},
        "steady_state_generation": st(steady["generation_time"]) if not steady.empty else {},
        "by_task": by_task,
        "avg_model_calls_per_customer": round(float(np.mean([r.get("n_model_calls", 0)
                                                             for r in records])), 2),
        "gpu_memory_trace": MEMORY_TRACE[-6:],
        "mrz_pages": dict(pd.Series([(r.get("mrz") or {}).get("mrz_candidate_page")
                                     for r in records]).value_counts(dropna=False)),
        "mrz_read_rate": round(float(np.mean([
            bool((r.get("identity_fields", {}).get("mrz") or {}).get("value"))
            for r in records])), 3),
        "kyc_status": dict(pd.Series([r.get("overall_kyc_status") for r in records])
                           .value_counts()),
    }
    print("=" * 70)
    print(f"customers {out['n_customers']}  pages {out['n_pages']}  "
          f"docs/s {out['documents_per_second']}  pages/s {out['pages_per_second']}")
    print(f"avg document time {out['avg_document_time']}s  "
          f"calls/customer {out['avg_model_calls_per_customer']}")
    print("-" * 70)
    print("share of measured time:", out["share_of_measured_time_pct"])
    print("steady-state inference:", out["steady_state_inference"], " (warm-up excluded)")
    print("-" * 70)
    for task, s in by_task.items():
        print(f"  {task:<24} n={s['n']:<4} inference mean={s['inference_time']['mean']}s "
              f"p95={s['inference_time']['p95']}s tokens={s['output_tokens']['mean']}")
    print("-" * 70)
    print("MRZ found on pages:", out["mrz_pages"], "| read rate:", out["mrz_read_rate"])
    print("KYC:", out["kyc_status"])
    print("=" * 70)
    (DIRS["reports"] / "benchmark_summary.json").write_text(
        json.dumps(out, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return out


print("CELL 33 ready: benchmark_pipeline()")

In [ ]:
# =========================================================================
# CELL 34 — RESULT AND DEBUG INSPECTION
# =========================================================================
def inspect_customer(customer_id: str, show_images: bool = False) -> Optional[Dict[str, Any]]:
    path = DIRS["results"] / f"{customer_id}.json"
    if not path.exists():
        print("no result for", customer_id)
        return None
    r = json.loads(path.read_text(encoding="utf-8"))
    print("=" * 78)
    print(f"{r['customer_id']}   KYC: {r.get('overall_kyc_status')}   "
          f"review: {r.get('needs_manual_review')}   {r.get('total_customer_time')}s   "
          f"calls={r.get('n_model_calls')}")
    if r.get("reasons"):
        print("reasons:", "; ".join(r["reasons"]))

    print("-" * 78)
    print("IDENTITY FIELDS (raw OCR, with provenance)")
    for f in IDENTITY_FIELDS:
        n = (r.get("identity_fields") or {}).get(f) or {}
        v = n.get("value")
        v = (v[:40] + "...") if isinstance(v, str) and len(v) > 43 else v
        print(f"  {f:<20} {str(v):<44} {str(n.get('confidence')):<6} "
              f"{str(n.get('confidence_band')):<10} p{n.get('source_page')} "
              f"{n.get('ocr_task') or ''}")
        if n.get("failure_reason"):
            print(f"      failure_reason: {n['failure_reason']}")

    mrz = r.get("mrz") or {}
    print("-" * 78)
    print("MRZ PIPELINE")
    print(f"  candidates examined on pages {mrz.get('candidate_pages')} "
          f"({mrz.get('candidates_examined')} total)")
    for k in ("mrz_detected", "mrz_candidate_page", "mrz_detection_method", "mrz_detection_score",
              "mrz_region", "mrz_source", "mrz_crop_width", "mrz_crop_height",
              "mrz_upscaled_char_height", "mrz_inference_time", "mrz_validation_status"):
        print(f"  {k:<26} {mrz.get(k)}")
    if mrz.get("mrz_candidate_metrics"):
        print("  discriminators            ", mrz["mrz_candidate_metrics"])
    for a in mrz.get("attempts", []):
        print(f"    attempt {a['attempt']} [{a['variant']}] {a['reason']}")
        print(f"      crop={a['crop']} char~{a['char_height_px']}px tokens={a['visual_tokens']} "
              f"ops={a['ops']} -> structurally_mrz={a['structurally_mrz']} "
              f"({a['validation_status']})")
    if mrz.get("rejected_non_mrz_transcription"):
        print("  REJECTED as not MRZ-shaped:", mrz["rejected_non_mrz_transcription"][:80])
    for k, c in (mrz.get("mrz_validation", {}) or {}).get("checks", {}).items():
        mark = "ok" if c.get("computed") == c.get("printed") else "MISMATCH"
        print(f"    check {k:<16} computed={c.get('computed')} printed={c.get('printed')} {mark}")
    if mrz.get("failure_reason"):
        print("  failure_reason:", mrz["failure_reason"])

    print("-" * 78)
    print("SECONDARY DOCUMENTS")
    for doc in DOCUMENTS_REQUIRED[1:]:
        res = (r.get("documents") or {}).get(doc)
        if not res:
            print(f"  {doc:<28} MISSING")
            continue
        print(f"  {doc:<28} {res.get('document_verification_status'):<10} "
              f"title={str(res.get('detected_document'))[:36]}")
        if doc == FATCA_DOC:
            print(f"      component_1 detected={res.get('component_1_detected')} "
                  f"header_verified={res.get('component_1_header_verified')} "
                  f"pages={res.get('component_1_page_count')}")
            print(f"      component_2 detected={res.get('component_2_detected')} "
                  f"title_verified={res.get('component_2_title_verified')}")
        for f in NAME_FIELDS:
            n = (res.get("fields") or {}).get(f) or {}
            print(f"      {f:<18} {str(n.get('value')):<24} {str(n.get('confidence_band')):<10} "
                  f"{str(n.get('text_type') or '-'):<12} {n.get('match_status') or '-'}")

    print("-" * 78)
    print("CROSS-DOCUMENT VERIFICATION")
    print(f"  {'document':<28}{'surname':<16}{'given names':<16}{'overall':<16}doc_type")
    for m in r.get("cross_document_verification", []):
        print(f"  {m['document']:<28}{str(m.get('surname_latin')):<16}"
              f"{str(m.get('given_names_latin')):<16}{str(m.get('overall')):<16}"
              f"{m.get('document_verification_status')}")
    db = (r.get("database_verification") or {}).get("database_comparison") or {}
    print(f"  DATABASE (db_id={r.get('database_verification', {}).get('db_id')})")
    for k, c in db.items():
        print(f"    {k:<16} doc={str(c.get('document_value'))[:24]:<26}"
              f"db={str(c.get('reference_value'))[:24]:<26}{c.get('status')} "
              f"({c.get('comparison_method')} score={c.get('similarity_score')})")
    print("timings:", r.get("timings_seconds"))

    if show_images:
        from IPython.display import display
        for img in sorted((DIRS["debug"] / str(customer_id)).rglob("*.png")):
            print(img.relative_to(DIRS["debug"]))
            im = Image.open(img)
            im.thumbnail((900, 900))
            display(im)
    return r


def diagnose_failures(records: List[Dict[str, Any]] = None) -> pd.DataFrame:
    """Why did a field come back null? Attribute each failure to a cause class:
    A bad render, B insufficient resolution, C bad crop, D preprocessing,
    E image input, F generation settings, G model limitation."""
    records = records if records is not None else load_results()
    rows = []
    for r in records:
        ident = (r.get("documents") or {}).get(IDENTITY_DOC) or {}
        pages = ident.get("pages", [])
        worst_text_h = min([p.get("page_quality", {}).get("text_height_px_in_view") or 99
                            for p in pages] or [99])
        native = [p.get("native_image_px") for p in pages if p.get("native_image_px")]
        for f, n in (r.get("identity_fields") or {}).items():
            if n.get("value") is not None:
                continue
            if any(p.get("status") == "PAGE_ERROR" for p in pages):
                cause = "A: page render/processing error"
            elif native and max(w for w, h in native) < 1200:
                cause = ("B: source PDF raster is only "
                         f"{max(w for w, h in native)}px wide -- no render DPI can help")
            elif worst_text_h < CFG.MIN_TEXT_HEIGHT:
                cause = f"B: glyphs ~{worst_text_h}px in view (< MIN_TEXT_HEIGHT)"
            elif f == "mrz" and (r.get("mrz") or {}).get("mrz_detected") is False:
                cause = "C: no MRZ-like region localised on any page"
            elif any(p.get("parse_ok_fields") is False for p in pages):
                cause = "F: model output was not parsable JSON"
            elif n.get("caps_applied") and "conflicting_pages" in n["caps_applied"]:
                cause = "G: pages disagreed; value withheld rather than guessed"
            else:
                cause = "G: region legible by measurement but the model did not read it"
            rows.append({"customer_id": r["customer_id"], "field": f,
                         "failure_reason": n.get("failure_reason") or
                                           (n.get("caps_applied") or [None])[0],
                         "probable_cause": cause,
                         "min_text_height_in_view": worst_text_h,
                         "native_raster_px": max((w for w, h in native), default=None),
                         "debug_dir": str(DIRS["debug"] / str(r["customer_id"]))})
    df = pd.DataFrame(rows)
    if len(df):
        df.to_csv(DIRS["reports"] / "failure_diagnostics.csv", index=False, encoding="utf-8-sig")
        print(df["probable_cause"].value_counts().to_string())
    return df


print("CELL 34 ready: inspect_customer(), diagnose_failures()")

## Driver

In [ ]:
# =========================================================================
# CELL 35 — DRIVER
# =========================================================================
WARMUP_STATS = warm_up(ENGINE, CFG)          # measured separately, excluded from steady state
print("warm-up:", WARMUP_STATS)

catalog_df = extract_zip(CFG.ZIP_PATH, CFG)
customers = discover_documents(catalog_df)
presence_df = build_presence_report(customers, catalog_df, CFG)
targets = select_targets(customers, CFG)

batch_df = run_batch(targets, ENGINE, DB, CFG)
records = load_results()
paths = write_outputs(records, CFG)
summary = benchmark_pipeline(records, CFG)
diag = diagnose_failures(records)

if len(batch_df):
    display(batch_df)
    inspect_customer(batch_df.iloc[0]["customer_id"])

## Technical explanation and self-audit

### 1. How Qwen3.8-27B-FP8 is loaded
`AutoConfig` confirms a vision tower exists, `AutoProcessor` is built once with
`min_pixels`/`max_pixels`, then the model class named in `config.json["architectures"]` is used
(falling back to `AutoModelForImageTextToText`, then `AutoModelForVision2Seq`) with
`quantization_config=FineGrainedFP8Config(...)`, `device_map="auto"` and a VRAM reserve so
accelerate cannot fill the card with weights and leave nothing for activations. Both objects are
process singletons — never reloaded per customer, document, page or retry.

### 2. How FineGrainedFP8Config is configured
**Verified against the installed package, and the spec's import path does not work:**
`from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config` raises `ImportError`
— that module contains `FP8Linear` and `replace_with_fp8_linear`, not the config class. The class
is defined in `transformers.utils.quantization_config` and re-exported at the `transformers` top
level, with signature `(activation_scheme="dynamic", weight_block_size=(128,128),
modules_to_not_convert=None)`. CELL 7 tries the specified path first, then the working ones, and
prints which resolved, so a future release that moves the class keeps working.

CELL 8 builds the config using only parameters the installed class accepts, and puts the **vision
tower and LM head in `modules_to_not_convert`**. That is an OCR-quality decision: the vision
encoder is what resolves MRZ glyphs and pen strokes, it is a small share of the parameters, and
quantising it trades the accuracy this pipeline exists to protect for memory it does not need.
Loading then **verifies FP8 actually applied** by counting `FP8Linear` modules and reading
`model.config.quantization_config`; if FP8 is requested and not present, it raises rather than
running silently on the non-quantised path.

### 3. Attention backend
`sdpa`, PyTorch's own fused attention. FlashAttention is never imported, probed or installed. If
the installed build rejects `sdpa` for this architecture the loader retries once with `eager` and
logs it; the backend actually in use is printed at startup and recorded in every result file.

### 4. H100 utilisation
`torch.inference_mode()` on every generate; greedy decoding with a brace-balance stop so calls end
when the JSON closes; `max_new_tokens` sized per task (512 / 160 / 160 / 128) rather than one large
cap; thinking mode disabled; decoded ids moved to CPU once and GPU references dropped;
`empty_cache()` only on OOM recovery, never per page. Memory is sampled at load, after warm-up,
during the batch and at the end.

### 5–8. MRZ
Every page is rendered and **every page runs MRZ candidate detection** — the pipeline never assumes
page 2. Detection is spatial and texture-based (blackhat → Sobel-x → closing → contours) and each
candidate carries measured discriminators: width fraction, aspect ratio, fill density, glyph-height
uniformity, character-pitch regularity, colour saturation and line count. Those are exactly what
separate an MRZ from a portrait (saturation, no line structure), Arabic identity text (height
uniformity), printed fields (width fraction), a document number elsewhere (aspect ratio) and
decoration (fill). Candidates are ranked across all pages, the winner's **entire region** is
cropped, and — the key resolution fix — the band is **re-rendered from the PDF at
`REGION_RENDER_DPI`** rather than upscaled from an already-downscaled page, then upscaled further
until characters reach `MRZ_MIN_CHAR_HEIGHT`. Only then does a dedicated MRZ OCR call run. After
OCR, `looks_like_mrz()` structurally confirms the transcription really is MRZ-shaped; a candidate
that yields non-MRZ text is rejected and recorded rather than accepted. Retries change the
preprocessing **variant** (`standard` → `aggressive` → `binarised`), each with a logged reason —
never the same input twice. ICAO check digits are computed as diagnostics only and can never
change a character.

### 9. Handwriting
FATCA and signature names get their own crops, re-rendered at high DPI, upscaled by
`HANDWRITING_UPSCALE_FACTOR`, with stroke-preserving preprocessing (no binarisation, which breaks
thin pen strokes). Bounded by `MAX_TARGETED_RETRIES`, with a different variant each time.

### 10–11. Verification isolation
Extraction completes independently per document before any comparison runs. Comparison produces
verdicts only — `MATCH / MISMATCH / UNCERTAIN / NOT_AVAILABLE` — and writes into `match_status`,
never into `value`. Unreadable is never a mismatch. Database use is restricted to the four
authorised columns by construction: `load_database()` selects them and drops everything else, so
no other column is reachable. Names compare as token multisets, so `BENALI MOHAMED` and
`MOHAMED BENALI` match and the observed order is reported.

### 12. Performance measurement
Per-stage timers (render, preprocess, ROI, MRZ detection, MRZ preprocessing, inference,
handwriting inference, cross-document, database), a per-call inference log with image dimensions
and input/output token counts, and a benchmark that **excludes the warm-up call** from steady-state
figures and breaks inference down by task.

---

### Why the previous version returned mostly null, and what changed

| Cause | Change |
|---|---|
| The prompt showed `{"value": null, "confidence": "unreadable"}` as the output template — copying it was a valid, zero-risk, entirely null completion | No null template is ever shown. Each prompt lists keys and a **filled** worked example |
| "One illegible character ⇒ the whole value is null" made refusal the safe move | The model marks individual illegible characters with `?` and Python applies the policy. `MOHA?ED` is captured instead of nothing, and it is visible in the audit trail |
| Schema extraction on a degraded scan asks for reading, layout and judgement at once | Two passes over the same pixels: PASS A transcribes lines verbatim, PASS B fills the schema, Python maps labelled lines to fields. **Agreement between the two is an objective confidence component** |
| Pages downscaled before the model saw them | Full-resolution render kept; tiles and ROIs re-rendered from the PDF at high DPI |
| MRZ read from the full page | Dedicated spatial detection on every page, own crop, own resolution target, own OCR task |

### Two things to verify on your data rather than take on trust

**`MAX_IMAGE_DIMENSION = 1600` is a starting point.** The MRZ no longer depends on it, but printed
field legibility does. `processing_log.csv` reports `text_height_in_view` per page; if it sits
below `MIN_TEXT_HEIGHT` on many pages, raise the dimension or the tile count before blaming the
model.

**The 180° orientation heuristic is the weakest component.** It reasons about ink position relative
to a text baseline — reliable on dense forms, weaker on sparse ID cards. `preprocessing.orientation`
records the method and margin per page; if `cv_projection` dominates with small margins, install
tesseract so the OSD tier takes over.

**When a field is still null**, run `diagnose_failures()`: it attributes each failure to a cause
class (A render, B resolution, C crop, D preprocessing, E image input, F generation, G model) using
the measurements already in the record, and points at the debug crop. Set `SAVE_DEBUG_IMAGES=True`
and look at what the model was actually shown — that single step answers most "why unreadable"
questions faster than any prompt change.